# Nội dung 3 — Thực nghiệm ACRF-RNN

**Mục tiêu:** Chạy mô hình ACRF-RNN với dữ liệu sau khi thực hiện Spark pipeline, so sánh **paper với hai phiên bản**.

| Phiên bản | Tiêu chí early stopping | Gradient update | Fix áp dụng |
|---|---|---|---|
| **VerA** — Code-faithful | `train_loss < loss_min` | 1 lần sau toàn bộ epoch | Fix 4-9 |
| **VerB** — Paper-faithful | `eval_ks > best_ks` | Mỗi `batch_train=4` bước | Fix 1–9 |

**Ý nghĩa:** So sánh VerA vs VerB định lượng tác động của mâu thuẫn M1 (early stopping) và M3 (gradient accumulation).

**Tiêu chí:** KS >= 35% trên ít nhất một năm kiểm thử; Recallm và G-mean trong khoảng +-5% so với bài báo.

## Giai đoạn 1 - Kiểm tra môi trường

### 1.1 Kiểm tra Python & GPU

In [80]:
!python --version
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,1), "GB")
else:
    print("Không có GPU - vào Runtime > Change runtime type > GPU")

Python 3.12.13
Torch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 14.6 GB


### 1.2 Tải thư viện

In [81]:
!pip install -q numpy pandas scipy scikit-learn tqdm

## Giai đoạn 2 - Clone repo và tạo thư mục

### 2.1 Clone repo tác giả

In [82]:
import os
os.chdir("/content")
print(os.getcwd())

/content


In [83]:
!rm -rf /content/ACRF-RNN
!git clone https://github.com/XNetLab/ACRF-RNN.git
!ls /content/ACRF-RNN/Code/

Cloning into 'ACRF-RNN'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 32 (delta 4), reused 32 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 3.45 MiB | 22.05 MiB/s, done.
Resolving deltas: 100% (4/4), done.
layer.py  main.py  model.py  util4.py


### 2.2 Tạo thư mục làm việc

In [84]:
import os
os.makedirs("/content/project_data", exist_ok=True)
os.makedirs("/content/project_outputs", exist_ok=True)
os.makedirs("/content/ACRF-RNN/Code/savemodel", exist_ok=True)
print("Thư mục sẵn sàng")
!ls /content/

Thư mục sẵn sàng
ACRF-RNN  drive  project_data  project_outputs	sample_data


## Giai đoạn 3 - Tải dữ liệu từ Google Drive

File `processed_flat.json` (output của Spark pipeline - Nội dung 2) đặt tại:
`/content/drive/MyDrive/ACRF-RNN/processed_flat.json`

### 3.1 Mount Google Drive

In [85]:
from google.colab import drive
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/ACRF-RNN'
print("Files trong Drive path:")
print(os.listdir(drive_path) if os.path.exists(drive_path) else "Không tìm thấy thư mục")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files trong Drive path:
['Nộp', 'fraud_project.zip', 'processed_flat.json', 'processed.zip', 'Spark_Pipeline.ipynb', 'acrf-rnn.zip', 'feature_analysis.zip', 'Spark_Temporal_Feature_Analysis.ipynb', 'ACRF_RNN.ipynb']


### 3.2 Copy và kiểm tra file JSON

In [86]:
JSON_SRC  = '/content/drive/MyDrive/ACRF-RNN/processed_flat.json'
JSON_DEST = '/content/project_data/processed_flat.json'

!cp "{JSON_SRC}" "{JSON_DEST}"

import json, os
size_mb = os.path.getsize(JSON_DEST) / 1024**2
with open(JSON_DEST) as f:
    meta = json.load(f)

print(f"File: {JSON_DEST} ({size_mb:.1f} MB)")
print(f"  format           : {meta['format']}")
print(f"  num_years        : {meta['num_years']}")
print(f"  num_companies    : {meta['num_companies_per_year']}")
print(f"  num_cols_per_row : {meta['num_columns_per_row']}")
print(f"  years            : {[d['year'] for d in meta['data']]}")
print(f"  last 3 cols      : {meta['columns'][-3:]}")

assert meta["num_columns_per_row"] == 209, "Số cột sai, kiểm tra lại pipeline"
assert meta["columns"][-1] == "label",    "Cột cuối phải là label"
print("\nFile JSON hợp lệ")

File: /content/project_data/processed_flat.json (20.5 MB)
  format           : main_py_compatible_year_major
  num_years        : 10
  num_companies    : 491
  num_cols_per_row : 209
  years            : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020]
  last 3 cols      : ['D000200000', 'sj_label', 'label']

File JSON hợp lệ


## Giai đoạn 4 - Import thư viện

In [87]:
import os, sys, json, random, re
import numpy as np
import pandas as pd
import torch
from torch import optim
from scipy.stats import ks_2samp
from sklearn.metrics import recall_score

## Giai đoạn 5 — Vá code

### Phân loại fix theo phiên bản

**Dùng chung (VerA và VerB)** — Fix lỗi CUDA crash:

| # | File | Dòng | Vấn đề | Fix |
|---|---|---|---|---|
| Fix 7 | `model.py` | 15 | `self.attentions=[...]` Python list → không chuyển lên GPU | `nn.ModuleList(...)` |
| Fix 8 | `layer.py` | 105 | `logits=torch.zeros(...)` tạo ở CPU; cộng với GPU tensor | Thêm `device=input_r.device` |
| Fix 9 | `layer.py` | 113–114 | `self.coef_revise` tạo ở CPU; nhân với GPU tensor | Thêm `.to(device)` |

**VerA only** — Fix crash Python 3.9+, giữ nguyên logic:

| # | File | Dòng | Vấn đề | Fix |
|---|---|---|---|---|
| Fix 4 | `main.py` | đầu file | Thiếu `import random`, `import os` | Thêm vào |
| Fix 5 | `main.py` | 54 | `args.dropout` chưa khai báo | Thêm `--dropout` |
| Fix 6 | `main.py` | 108 | `random.shuffle(seq, random=r)` lỗi Python ≥ 3.9 | Bỏ `random=` |

**VerB thêm** — Fix logic + đọc JSON:

| # | File | Dòng | Vấn đề | Fix |
|---|---|---|---|---|
| Fix 1 | `main.py` | 81 | `np.squeeze(axis=2)` lỗi khi JSON đã là `[T,N,209]` | Bỏ squeeze |
| Fix 2 | `main.py` | 89 | `test_line=test_year-2021-1` hardcode | `years.index(test_year)` |
| Fix 3 | `main.py` | 133,167 | `plt_output` chưa định nghĩa | Xóa các dòng tham chiếu |

### 5.1 Vá `layer.py` — Fix 8, 9

**Fix 8** (`Attention.forward()`, dòng 105):
```python
# TRƯỚC (lỗi): logits tạo ở CPU, cộng với f_1/f_2 ở GPU → crash
logits = torch.zeros(num_stock, num_stock, dtype=input_r.dtype)

# SAU: thêm device=device
logits = torch.zeros(num_stock, num_stock, dtype=input_r.dtype, device=device)
```

**Fix 9** (`Attention.forward()`, dòng 113-114):
```python
# TRƯỚC (lỗi): coef_revise tạo ở CPU, nhân với coefs ở GPU → crash
self.coef_revise = torch.zeros(...) + 1.0 - torch.eye(...)

# SAU: thêm .to(device)
self.coef_revise = (torch.zeros(...) + 1.0 - torch.eye(...)).to(device)
```

In [88]:
# layer.py — Fix 8 (logits device) + Fix 9 (coef_revise device)
fixed_layer = r'''
import torch.nn.functional as F
from torch import nn
import torch
from util4 import *


class CRF_Att(nn.Module):

    def __init__(self, input_dim, output_dim, num_iters):
        super(CRF_Att, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_iters = num_iters
        self.al   = nn.Parameter(torch.zeros(1).type(torch.FloatTensor))
        self.beta = nn.Parameter(torch.zeros(1).type(torch.FloatTensor))

    def forward(self, inputs, similarity):
        support   = similarity
        normalize = torch.sum(support, dim=1)
        normalize = torch.Tensor.repeat(torch.unsqueeze(normalize, -1), [1, self.input_dim])
        al   = torch.exp(self.al)
        beta = torch.exp(self.beta)
        output = inputs
        iters  = torch.tensor(0)
        cond   = lambda iters, num_iters: torch.le(iters, self.num_iters)
        while cond(iters, output):
            output = (inputs * beta + (dot(support, output) + output) * al) \
                     / (beta + normalize * al + al)
            iters = torch.add(iters, 1)
        return output


class Linear(nn.Module):
    def __init__(self, num_nodes, input_size, hidden_size, bias=True):
        super(Linear, self).__init__()
        self.bias = bias
        self.W = nn.Parameter(torch.zeros(num_nodes, input_size, hidden_size))
        self.b = nn.Parameter(torch.zeros(num_nodes, hidden_size))
        self.reset_parameters()

    def reset_parameters(self):
        reset_parameters(self.named_parameters)

    def forward(self, x):
        output = torch.bmm(x.unsqueeze(1), self.W)
        output = output.squeeze(1)
        if self.bias:
            output = output + self.b
        return output


class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, bias=True):
        super(GRUModel, self).__init__()
        self.GRU_layer1 = nn.GRU(input_size=input_dim,    hidden_size=2 * hidden_dim)
        self.GRU_layer2 = nn.GRU(input_size=2 * hidden_dim, hidden_size=hidden_dim)
        self.hidden_dim = hidden_dim
        self.hidden = None
        self.reset_parameters()

    def reset_parameters(self):
        reset_parameters(self.named_parameters)

    def forward(self, x):
        x, hidden1 = self.GRU_layer1(x)
        x, hidden2 = self.GRU_layer2(x)
        hidden = hidden2.squeeze(0)
        return hidden


class Attention(nn.Module):
    def __init__(self, in_features, out_features, alpha, num_company, residual=False):
        super(Attention, self).__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.alpha        = alpha
        self.num_company  = num_company
        self.seq_transformation_r = nn.Conv1d(in_features, out_features, kernel_size=1, stride=1, bias=False)
        self.f_1 = nn.Conv1d(out_features, 1, kernel_size=1, stride=1)
        self.f_2 = nn.Conv1d(out_features, 1, kernel_size=1, stride=1)
        self.W_static    = nn.Parameter(torch.zeros(self.num_company, self.num_company).type(torch.FloatTensor), requires_grad=True)
        self.w_1         = nn.Conv1d(in_features, out_features, kernel_size=1, stride=1)
        self.w_2         = nn.Conv1d(in_features, out_features, kernel_size=1, stride=1)
        self.coef_revise = False
        self.leakyrelu   = nn.LeakyReLU(self.alpha)

    def forward(self, input_r, relation_static=None):
        device    = input_r.device                                    # lấy device từ input
        num_stock = input_r.shape[0]
        seq_r     = torch.transpose(input_r, 0, 1).unsqueeze(0)
        # Fix 8: thêm device=device để logits cùng device với f_1, f_2
        logits    = torch.zeros(num_stock, num_stock, dtype=input_r.dtype, device=device)
        seq_fts_r = self.seq_transformation_r(seq_r)
        f_1 = self.f_1(seq_fts_r)
        f_2 = self.f_2(seq_fts_r)
        logits += (torch.transpose(f_1, 2, 1) + f_2).squeeze(0)
        if relation_static is not None:
            logits += torch.mul(relation_static, self.W_static)
        coefs = self.leakyrelu(logits)
        # Fix 9: thêm .to(device) để coef_revise cùng device với coefs
        if not isinstance(self.coef_revise, torch.Tensor):
            self.coef_revise = (
                torch.zeros(self.num_company, self.num_company)
                + 1.0
                - torch.eye(self.num_company, self.num_company)
            ).to(device)
        else:
            self.coef_revise = self.coef_revise.to(device)
        coefs_eye = coefs.mul(self.coef_revise)
        return coefs_eye


def myPool(type, crf_embeddings):
    head = len(crf_embeddings)
    company_num, feature_num = crf_embeddings[0].size()
    stacked_tensor = torch.stack(crf_embeddings, dim=1)
    pool_embedding = stacked_tensor.view(-1, feature_num)
    pool_embedding = pool_embedding.t()
    if type == "MAX":
        pool = nn.MaxPool1d(kernel_size=head, stride=head)
        pooled_embedding = pool(pool_embedding).t()
    elif type == "AVG":
        pool = nn.AvgPool1d(kernel_size=head, stride=head)
        pooled_embedding = pool(pool_embedding).t()
    elif type == "SUM":
        stacked_tensor2  = torch.stack(crf_embeddings, dim=0)
        pooled_embedding = torch.sum(stacked_tensor2, dim=0)
    return pooled_embedding
'''

with open("/content/ACRF-RNN/Code/layer.py", "w", encoding="utf-8") as f:
    f.write(fixed_layer)
print("Đã ghi /content/ACRF-RNN/Code/layer.py")
!wc -l /content/ACRF-RNN/Code/layer.py

Đã ghi /content/ACRF-RNN/Code/layer.py
129 /content/ACRF-RNN/Code/layer.py


### 5.2 Vá `model.py` — Fix 7

**Fix 7** (`AC_RNN.__init__()`, dòng 15):
```python
# TRƯỚC (lỗi): Python list — model.to(DEVICE) không chuyển Attention lên GPU
self.attentions = [Attention(...) for _ in range(heads_att)]

# SAU: nn.ModuleList — model.to(DEVICE) chuyển toàn bộ module lên GPU
self.attentions = nn.ModuleList([Attention(...) for _ in range(heads_att)])
```

In [89]:
# model.py —Fix 7 (nn.ModuleList)
fixed_model = r'''
from layer import *
from torch import nn


class AC_RNN(nn.Module):
    def __init__(self, num_company, d_feature, d_hidden, hidn_rnn,
                 heads_att, hidn_att, crf_iters, alpha, pool_type):
        super(AC_RNN, self).__init__()
        self.heads_att   = heads_att
        self.alpha       = alpha
        self.num_company = num_company
        self.d_feature   = d_feature
        self.d_hidden    = d_hidden
        self.hidden_rnn  = hidn_rnn
        self.hidn_att    = hidn_att
        self.GRUs        = GRUModel(d_feature, hidn_rnn)
        # Fix 7: đổi từ Python list sang nn.ModuleList
        # nn.ModuleList đảm bảo model.to(DEVICE) sẽ chuyển tất cả
        # Attention modules (và Conv1d bên trong) lên dùng device
        self.attentions  = nn.ModuleList([
            Attention(hidn_rnn, hidn_att, num_company=self.num_company, alpha=self.alpha)
            for _ in range(heads_att)
        ])
        self.crf         = CRF_Att(hidn_rnn, hidn_rnn, crf_iters)
        self.pool_type   = pool_type
        self.linear      = Linear(num_company, hidn_rnn, 2, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        reset_parameters(self.named_parameters)

    def forward(self, x):
        x            = self.GRUs(x)
        x            = F.dropout(x, self.drop_out)
        ss           = [att(x) for att in self.attentions]
        crf_embeddings = []
        for i in ss:
            crf_x_attention = self.crf(x, i)
            crf_embeddings.append(crf_x_attention)
        crf_x      = myPool(self.pool_type, crf_embeddings)
        crf_output = F.elu(self.linear(crf_x))
        crf_output = F.softmax(crf_output, dim=1)
        return crf_output


class FocalLoss(nn.Module):
    def __init__(self, weight=None, reduction="mean", gamma=0, eps=1e-7):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.eps   = eps
        self.ce    = torch.nn.CrossEntropyLoss(weight=weight, reduction=reduction)

    def forward(self, input, target):
        logp = self.ce(input, target)
        p    = torch.exp(-logp)
        loss = (1 - p) ** self.gamma * logp
        return loss.mean()
'''

with open("/content/ACRF-RNN/Code/model.py", "w", encoding="utf-8") as f:
    f.write(fixed_model)
print("Đã ghi /content/ACRF-RNN/Code/model.py")
!wc -l /content/ACRF-RNN/Code/model.py

Đã ghi /content/ACRF-RNN/Code/model.py
58 /content/ACRF-RNN/Code/model.py


### 5.2.1 Tạo `model_attention.py` để trích xuất Attention Graph

Không sửa trực tiếp `model.py` cũ để tránh làm hỏng luồng train/test ban đầu.  
Thay vào đó, tạo thêm file `model_attention.py`, giữ nguyên kiến trúc `AC_RNN` nhưng bổ sung tham số `return_attention`.

- Nếu `return_attention=False`: model trả output như cũ.
- Nếu `return_attention=True`: model trả thêm `attention_stack`.

`attention_stack` là tensor chứa các ma trận attention của nhiều head:

```text
attention_stack shape = [heads_att, num_company, num_company]

In [90]:
# model_attention.py

model_attention_code = r'''
from layer import *
from torch import nn
import torch
import torch.nn.functional as F


class AC_RNN(nn.Module):
    def __init__(self, num_company, d_feature, d_hidden, hidn_rnn,
                 heads_att, hidn_att, crf_iters, alpha, pool_type):
        super(AC_RNN, self).__init__()

        self.heads_att   = heads_att
        self.alpha       = alpha
        self.num_company = num_company
        self.d_feature   = d_feature
        self.d_hidden    = d_hidden
        self.hidden_rnn  = hidn_rnn
        self.hidn_att    = hidn_att

        self.GRUs = GRUModel(d_feature, hidn_rnn)

        self.attentions = nn.ModuleList([
            Attention(
                hidn_rnn,
                hidn_att,
                num_company=self.num_company,
                alpha=self.alpha
            )
            for _ in range(heads_att)
        ])

        self.crf       = CRF_Att(hidn_rnn, hidn_rnn, crf_iters)
        self.pool_type = pool_type
        self.linear    = Linear(num_company, hidn_rnn, 2, bias=True)

        # Giữ tương thích với main_vera.py cũ
        # vì main_vera.py sẽ gán model.drop_out = args.dropout sau khi tạo model.
        self.drop_out = 0.0

        self.reset_parameters()

    def reset_parameters(self):
        reset_parameters(self.named_parameters)

    def forward(self, x, return_attention=False):
        """
        x shape dự kiến:
            [rnn_len, num_company, d_feature]

        return_attention=False:
            return crf_output

        return_attention=True:
            return crf_output, attention_stack

        attention_stack shape:
            [heads_att, num_company, num_company]
        """

        # 1. GRU học embedding theo thời gian
        x = self.GRUs(x)

        # 2. Dropout giống model cũ
        x = F.dropout(x, self.drop_out, training=self.training)

        # 3. Tính attention matrix cho từng head
        attention_list = [att(x) for att in self.attentions]

        # 4. Đưa từng attention matrix vào CRF
        crf_embeddings = []
        for att_matrix in attention_list:
            crf_x_attention = self.crf(x, att_matrix)
            crf_embeddings.append(crf_x_attention)

        # 5. Pool các head
        crf_x = myPool(self.pool_type, crf_embeddings)

        # 6. Classifier
        crf_output = F.elu(self.linear(crf_x))
        crf_output = F.softmax(crf_output, dim=1)

        # 7. Nếu cần attention thì trả thêm attention_stack
        if return_attention:
            attention_stack = torch.stack(attention_list, dim=0)
            return crf_output, attention_stack

        return crf_output


class FocalLoss(nn.Module):
    def __init__(self, weight=None, reduction="mean", gamma=0, eps=1e-7):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.eps   = eps
        self.ce    = torch.nn.CrossEntropyLoss(weight=weight, reduction=reduction)

    def forward(self, input, target):
        logp = self.ce(input, target)
        p    = torch.exp(-logp)
        loss = (1 - p) ** self.gamma * logp
        return loss.mean()
'''

with open("/content/ACRF-RNN/Code/model_attention.py", "w", encoding="utf-8") as f:
    f.write(model_attention_code)

print("Đã tạo /content/ACRF-RNN/Code/model_attention.py")
!wc -l /content/ACRF-RNN/Code/model_attention.py
!head -30 /content/ACRF-RNN/Code/model_attention.py

Đã tạo /content/ACRF-RNN/Code/model_attention.py
102 /content/ACRF-RNN/Code/model_attention.py

from layer import *
from torch import nn
import torch
import torch.nn.functional as F


class AC_RNN(nn.Module):
    def __init__(self, num_company, d_feature, d_hidden, hidn_rnn,
                 heads_att, hidn_att, crf_iters, alpha, pool_type):
        super(AC_RNN, self).__init__()

        self.heads_att   = heads_att
        self.alpha       = alpha
        self.num_company = num_company
        self.d_feature   = d_feature
        self.d_hidden    = d_hidden
        self.hidden_rnn  = hidn_rnn
        self.hidn_att    = hidn_att

        self.GRUs = GRUModel(d_feature, hidn_rnn)

        self.attentions = nn.ModuleList([
            Attention(
                hidn_rnn,
                hidn_att,
                num_company=self.num_company,
                alpha=self.alpha
            )
            for _ in range(heads_att)


### 5.2.2 Chuẩn bị export attention weights từ dữ liệu thật

Sau khi `model_attention.py` đã trả được `attention_stack`, tiếp theo là chạy model trên dữ liệu thật `processed_flat.json`.

Mục tiêu của bước này:

1. Đọc dữ liệu thật đã xử lý.
2. Kiểm tra shape của `x`, `y`, `years`.
3. Xác định số công ty, số năm, số feature.
4. Kiểm tra checkpoint model đã train.
5. Sau đó mới export `attention_weights.csv`.

Lưu ý: Attention graph phải được lấy từ model đã train. Nếu chưa load checkpoint, attention chỉ là trọng số ngẫu nhiên và chưa có ý nghĩa phân tích.

In [91]:
# Load processed_flat.json và tách x_np, y_np theo format main_py_compatible_year_major

import os
import json
import numpy as np
import pandas as pd
import torch

PROJECT_DATA_PATH = "/content/project_data/processed_flat.json"
CODE_DIR = "/content/ACRF-RNN/Code"
ATTENTION_OUTPUT_DIR = "/content/project_outputs/attention_graph"

os.makedirs(ATTENTION_OUTPUT_DIR, exist_ok=True)

with open(PROJECT_DATA_PATH, "r", encoding="utf-8") as f:
    flat_data = json.load(f)

assert flat_data["format"] == "main_py_compatible_year_major"

data_items = flat_data["data"]
columns = flat_data["columns"]

years = []
all_rows = []
all_symbols = []

for item in data_items:
    years.append(int(item["year"]))
    all_symbols.append(item["symbols"])
    all_rows.append(np.array(item["rows"], dtype=np.float32))

raw_np = np.stack(all_rows, axis=0)

label_col = "label"
label_idx = columns.index(label_col)

y_np = raw_np[:, :, label_idx].astype(np.int64)

feature_indices = [i for i in range(len(columns)) if i != label_idx]
x_np = raw_np[:, :, feature_indices].astype(np.float32)
feature_columns = [columns[i] for i in feature_indices]

num_years = x_np.shape[0]
num_company = x_np.shape[1]
d_feature = x_np.shape[2]

print("raw_np shape:", raw_np.shape)
print("x_np shape:", x_np.shape)
print("y_np shape:", y_np.shape)
print("years:", years)
print("Label values:", np.unique(y_np))
print("num feature columns:", len(feature_columns))

assert num_years == 10
assert num_company == 491
assert d_feature == 208
assert set(np.unique(y_np)).issubset({0, 1})

print("Dữ liệu thật hợp lệ để chuẩn bị export attention.")

raw_np shape: (10, 491, 209)
x_np shape: (10, 491, 208)
y_np shape: (10, 491)
years: [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020]
Label values: [0 1]
num feature columns: 208
Dữ liệu thật hợp lệ để chuẩn bị export attention.


In [92]:
import glob
import os

checkpoint_candidates = []

search_dirs = [
    "/content/ACRF-RNN",
    "/content/ACRF-RNN/Code",
    "/content/project_outputs",
    "/content/project_outputs/checkpoints",
    "/content/drive/MyDrive",
]

patterns = ["*.pt", "*.pth", "*.ckpt", "*.pkl"]

for d in search_dirs:
    if os.path.exists(d):
        for p in patterns:
            checkpoint_candidates.extend(
                glob.glob(os.path.join(d, "**", p), recursive=True)
            )

checkpoint_candidates = sorted(set(checkpoint_candidates))

print("Found checkpoint candidates:")
for i, ckpt in enumerate(checkpoint_candidates[:50]):
    print(i, ckpt)

print("Total checkpoints found:", len(checkpoint_candidates))

Found checkpoint candidates:
0 /content/project_outputs/checkpoints/vera_best_test_year_2018.pt
Total checkpoints found: 1


### 5.2.3 Tạo `main_vera_ckpt.py` để lưu checkpoint model

Hiện tại `main_vera.py` chỉ train và in log, chưa lưu trọng số model.  
Để trích xuất Attention Graph có ý nghĩa, cần lấy attention từ model đã train, không lấy từ model random.

Ta tạo thêm file `main_vera_ckpt.py` dựa trên `main_vera.py` cũ.  
Logic train/evaluation của VerA được giữ nguyên, chỉ bổ sung thao tác lưu checkpoint tại epoch có `train_loss` tốt nhất.

Checkpoint sau khi train sẽ được lưu vào:

```text
/content/project_outputs/checkpoints/vera_best_test_year_2018.pt
/content/project_outputs/checkpoints/vera_best_test_year_2019.pt
/content/project_outputs/checkpoints/vera_best_test_year_2020.pt
```

In [112]:
# Tạo main_vera_ckpt.py từ main_vera.py và bổ sung lưu checkpoint

import os

MAIN_VERA_PATH = "/content/ACRF-RNN/Code/main_vera.py"
MAIN_VERA_CKPT_PATH = "/content/ACRF-RNN/Code/main_vera_ckpt.py"

assert os.path.exists(MAIN_VERA_PATH), f"Không tìm thấy {MAIN_VERA_PATH}"

with open(MAIN_VERA_PATH, "r", encoding="utf-8") as f:
    code = f.read()

# 1. Thêm argument checkpoint directory nếu chưa có
old_arg = 'parser.add_argument("--dropout",     type=float, default=0.0)'
new_arg = '''parser.add_argument("--dropout",     type=float, default=0.0)
parser.add_argument("--ckpt-dir",    type=str,   default="/content/project_outputs/checkpoints")'''

if '--ckpt-dir' not in code:
    if old_arg not in code:
        raise ValueError("Không tìm thấy vị trí parser.add_argument('--dropout', ...) để chèn --ckpt-dir")
    code = code.replace(old_arg, new_arg)

# 2. Tạo thư mục checkpoint sau khi parse args
old_parse = '''args        = parser.parse_args()
rnn_len     = args.rnn_length'''

new_parse = '''args        = parser.parse_args()
os.makedirs(args.ckpt_dir, exist_ok=True)

rnn_len     = args.rnn_length'''

if 'os.makedirs(args.ckpt_dir, exist_ok=True)' not in code:
    if old_parse not in code:
        raise ValueError("Không tìm thấy vị trí parse_args để chèn os.makedirs(args.ckpt_dir, exist_ok=True)")
    code = code.replace(old_parse, new_parse)

# 3. Thêm đường dẫn checkpoint sau khi tạo optimizer
old_optimizer = '''optimizer = optim.Adam(model.parameters(), lr=args.lr)
    print(f"Model params: {sum(p.numel() for p in model.parameters()):,}\\\\n")'''

new_optimizer = '''optimizer = optim.Adam(model.parameters(), lr=args.lr)
    ckpt_path = os.path.join(args.ckpt_dir, f"vera_best_test_year_{test_year}.pt")
    print(f"Model params: {sum(p.numel() for p in model.parameters()):,}\\\\n")
    print(f"Checkpoint will be saved to: {ckpt_path}\\\\n")'''

if 'ckpt_path = os.path.join(args.ckpt_dir' not in code:
    if old_optimizer not in code:
        raise ValueError("Không tìm thấy vị trí optimizer để chèn ckpt_path")
    code = code.replace(old_optimizer, new_optimizer)

# 4. Khi gặp best train_loss thì lưu checkpoint
old_best_block = '''if tr_loss < loss_min:
            loss_min   = tr_loss
            best_epoch = epoch
            best_m     = ev
            wait_count = 0
            is_best    = "best"'''

new_best_block = '''if tr_loss < loss_min:
            loss_min   = tr_loss
            best_epoch = epoch
            best_m     = ev
            wait_count = 0
            is_best    = "best"

            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "test_year": test_year,
                "best_epoch": best_epoch,
                "best_train_loss": loss_min,
                "best_eval_metrics": best_m,
                "args": vars(args),
                "num_company": num_company,
                "d_feature": d_feature
            }, ckpt_path)'''

if '"model_state_dict": model.state_dict()' not in code:
    if old_best_block not in code:
        raise ValueError("Không tìm thấy block if tr_loss < loss_min để chèn torch.save")
    code = code.replace(old_best_block, new_best_block)

with open(MAIN_VERA_CKPT_PATH, "w", encoding="utf-8") as f:
    f.write(code)

print("Đã tạo:", MAIN_VERA_CKPT_PATH)
print("Kiểm tra các dòng checkpoint:")
!grep -n "ckpt\\|torch.save\\|model_state_dict" /content/ACRF-RNN/Code/main_vera_ckpt.py | head -30


Đã tạo: /content/ACRF-RNN/Code/main_vera_ckpt.py
Kiểm tra các dòng checkpoint:
31:parser.add_argument("--ckpt-dir",    type=str,   default="/content/project_outputs/checkpoints")
35:os.makedirs(args.ckpt_dir, exist_ok=True)
170:    ckpt_path = os.path.join(args.ckpt_dir, f"vera_best_test_year_{test_year}.pt")
172:    print(f"Checkpoint will be saved to: {ckpt_path}\\n")
199:            torch.save({
200:                "model_state_dict": model.state_dict(),
209:            }, ckpt_path)


In [113]:
# Chạy train để tạo checkpoint cho 2018, 2019, 2020

import os
import subprocess
from pathlib import Path

TEST_YEARS = [2018, 2019, 2020]
CKPT_DIR = "/content/project_outputs/checkpoints"
LOG_DIR = "/content/project_outputs"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Nếu muốn train lại từ đầu và ghi đè checkpoint cũ, đổi thành True
FORCE_RETRAIN = True

for year in TEST_YEARS:
    ckpt_path = f"{CKPT_DIR}/vera_best_test_year_{year}.pt"
    log_path = f"{LOG_DIR}/run_vera_ckpt_{year}.log"

    if os.path.exists(ckpt_path) and not FORCE_RETRAIN:
        print(f"[SKIP] Đã có checkpoint {year}: {ckpt_path}")
        continue

    print("=" * 80)
    print(f"TRAIN CHECKPOINT FOR TEST_YEAR = {year}")
    print("=" * 80)

    cmd = [
        "python", "main_vera_ckpt.py",
        "--data-path", "/content/project_data/processed_flat.json",
        "--test-year", str(year),
        "--max-epoch", "800",
        "--wait-epoch", "150",
        "--device", "0",
        "--ckpt-dir", CKPT_DIR
    ]

    with open(log_path, "w", encoding="utf-8") as log_f:
        process = subprocess.Popen(
            cmd,
            cwd="/content/ACRF-RNN/Code",
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )

        for line in process.stdout:
            print(line, end="")
            log_f.write(line)

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(f"Train checkpoint năm {year} bị lỗi. Xem log: {log_path}")

    print(f"Đã train xong năm {year}. Log: {log_path}")


TRAIN CHECKPOINT FOR TEST_YEAR = 2018
\nACRF-RNN VerA | test_year=2018 | device=cuda:0
Loaded: 10 years | x=(10, 491, 208)
Years : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020]
Train : [2010, 2011, 2012, 2013, 2014, 2015, 2017] (7 years)
Eval  : [2012, 2013, 2014, 2015, 2017, 2018]
Train labels: fraud=860, benign=2577
Model params: 972,937\n
Checkpoint will be saved to: /content/project_outputs/checkpoints/vera_best_test_year_2018.pt\n
  Ep |    Loss |  TrAcc |  EvAcc |  Rcl_m |     KS |     GM |
-------------------------------------------------------------
   0 |  0.4153 | 0.5580 | 0.4725 | 0.4469 | 0.1063 | 0.4437 | best
  --> [  0.1%] ep 1/800 | elapsed 0s | ETA 368s | best loss 0.4153 @ ep0 | wait 0/150
   1 |  0.4404 | 0.5204 | 0.5214 | 0.4926 | 0.0147 | 0.4890 | 
   2 |  0.4085 | 0.4939 | 0.5356 | 0.5048 | 0.0096 | 0.5008 | best
  --> [  0.4%] ep 3/800 | elapsed 1s | ETA 151s | best loss 0.4085 @ ep2 | wait 0/150
   3 |  0.3919 | 0.5458 | 0.5743 | 0.5335 | 0.0670 |

In [114]:
# Kiểm tra checkpoint sau khi train

import os
import glob
import torch

TEST_YEARS = [2018, 2019, 2020]
CKPT_DIR = "/content/project_outputs/checkpoints"

ckpt_files = sorted(glob.glob(f"{CKPT_DIR}/*.pt"))

print("Checkpoint files:")
for f in ckpt_files:
    print(f, round(os.path.getsize(f) / 1024 / 1024, 2), "MB")

print("\nKiểm tra từng checkpoint:")
for year in TEST_YEARS:
    target_ckpt = f"{CKPT_DIR}/vera_best_test_year_{year}.pt"
    print("\n" + "=" * 80)
    print("target:", target_ckpt)
    print("exists:", os.path.exists(target_ckpt))

    if os.path.exists(target_ckpt):
        ckpt = torch.load(target_ckpt, map_location="cpu", weights_only=False)

        print("checkpoint keys:", ckpt.keys())
        print("test_year:", ckpt.get("test_year"))
        print("best_epoch:", ckpt.get("best_epoch"))
        print("best_train_loss:", ckpt.get("best_train_loss"))
        print("num_company:", ckpt.get("num_company"))
        print("d_feature:", ckpt.get("d_feature"))

        if "best_eval_metrics" in ckpt:
            print("best_eval_metrics:", ckpt["best_eval_metrics"])

        if "args" in ckpt:
            print("args keys:", ckpt["args"].keys())


Checkpoint files:
/content/project_outputs/checkpoints/vera_best_test_year_2018.pt 5.56 MB
/content/project_outputs/checkpoints/vera_best_test_year_2019.pt 5.56 MB
/content/project_outputs/checkpoints/vera_best_test_year_2020.pt 5.56 MB

Kiểm tra từng checkpoint:

target: /content/project_outputs/checkpoints/vera_best_test_year_2018.pt
exists: True
checkpoint keys: dict_keys(['model_state_dict', 'optimizer_state_dict', 'test_year', 'best_epoch', 'best_train_loss', 'best_eval_metrics', 'args', 'num_company', 'd_feature'])
test_year: 2018
best_epoch: 22
best_train_loss: 0.09797326475381851
num_company: 491
d_feature: 208
best_eval_metrics: [np.float64(0.7535641547861507), 0.6953686077701826, np.float64(0.39073721554036517), np.float64(0.6848368206879467)]
args keys: dict_keys(['data_path', 'rnn_length', 'heads_att', 'hidn_att', 'hidn_rnn', 'crf_iter', 'max_epoch', 'wait_epoch', 'device', 'alpha', 'batch_train', 'clip', 'seed', 'lr', 'gamma', 'n_weight', 'f_weight', 'pool_type', 'test_yea

### 5.2.4 Load checkpoint vào `model_attention.py` và lấy attention theo từng năm

Sau khi đã có checkpoint cho các năm kiểm thử, bước này khởi tạo lại `AC_RNN` từ `model_attention.py`, nạp `model_state_dict`, rồi chạy model với `return_attention=True`.

Mục tiêu:
- Đảm bảo kiến trúc `model_attention.py` khớp với checkpoint.
- Lấy được `attention_stack` cho từng năm: 2018, 2019, 2020.
- Attention lấy ra là attention của model đã train, không còn là trọng số random.


In [115]:
# Load checkpoint vào model_attention.py và lấy attention_stack cho từng năm

import sys
import os
import torch

sys.path.append("/content/ACRF-RNN/Code")

from model_attention import AC_RNN

TEST_YEARS = [2018, 2019, 2020]
CKPT_DIR = "/content/project_outputs/checkpoints"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

pool_type_map = {
    1: "MAX",
    2: "MEAN",
    3: "SUM",
    "1": "MAX",
    "2": "MEAN",
    "3": "SUM",
    "max": "MAX",
    "mean": "MEAN",
    "avg": "MEAN",
    "sum": "SUM",
    "MAX": "MAX",
    "MEAN": "MEAN",
    "AVG": "MEAN",
    "SUM": "SUM",
}

def load_attention_model_for_year(test_year):
    ckpt_path = f"{CKPT_DIR}/vera_best_test_year_{test_year}.pt"
    assert os.path.exists(ckpt_path), f"Không tìm thấy checkpoint: {ckpt_path}"

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    args = ckpt["args"]

    pool_type_for_model = pool_type_map.get(args["pool_type"])
    if pool_type_for_model is None:
        raise ValueError(f"pool_type không hợp lệ: {args['pool_type']}")

    model = AC_RNN(
        num_company=ckpt["num_company"],
        d_feature=ckpt["d_feature"],
        d_hidden=ckpt["d_feature"],
        hidn_rnn=args["hidn_rnn"],
        heads_att=args["heads_att"],
        hidn_att=args["hidn_att"],
        crf_iters=args["crf_iter"],
        alpha=args["alpha"],
        pool_type=pool_type_for_model
    )

    model.drop_out = args.get("dropout", 0.0)

    missing_keys, unexpected_keys = model.load_state_dict(
        ckpt["model_state_dict"],
        strict=False
    )

    if len(missing_keys) > 0 or len(unexpected_keys) > 0:
        print("missing_keys:", missing_keys)
        print("unexpected_keys:", unexpected_keys)
        raise RuntimeError(f"Checkpoint năm {test_year} không khớp model_attention.py")

    model.to(device)
    model.eval()

    return model, ckpt, args


def get_attention_for_year(test_year):
    model, ckpt, args = load_attention_model_for_year(test_year)

    test_year_idx = years.index(test_year)
    rnn_len = args["rnn_length"]

    # Giữ đúng cách đang kiểm thử: lấy window kết thúc tại test_year
    start_idx = test_year_idx - rnn_len + 1
    end_idx = test_year_idx + 1

    if start_idx < 0:
        raise ValueError(f"Không đủ dữ liệu trước test_year={test_year} để tạo window.")

    x_window_np = x_np[start_idx:end_idx]
    x_window = torch.tensor(x_window_np, dtype=torch.float32, device=device)

    with torch.no_grad():
        pred, attention_stack = model(x_window, return_attention=True)

    info = {
        "test_year": test_year,
        "test_year_idx": test_year_idx,
        "window_years": years[start_idx:end_idx],
        "rnn_len": rnn_len,
        "checkpoint_best_epoch": ckpt.get("best_epoch"),
        "checkpoint_best_train_loss": ckpt.get("best_train_loss")
    }

    return pred, attention_stack, info



device: cuda:0


In [116]:
# Test lấy attention_stack cho 2018, 2019, 2020

attention_results = {}

for year in TEST_YEARS:
    pred, attention_stack, info = get_attention_for_year(year)
    attention_results[year] = {
        "pred": pred,
        "attention_stack": attention_stack,
        "info": info
    }

    print("\n" + "=" * 80)
    print("year:", year)
    print("window years:", info["window_years"])
    print("best_epoch:", info["checkpoint_best_epoch"])
    print("best_train_loss:", info["checkpoint_best_train_loss"])
    print("pred shape:", pred.shape)
    print("attention_stack shape:", attention_stack.shape)
    print("attention min:", attention_stack.min().item())
    print("attention max:", attention_stack.max().item())
    print("attention has NaN:", torch.isnan(attention_stack).any().item())



year: 2018
window years: [2013, 2014, 2015, 2017, 2018]
best_epoch: 22
best_train_loss: 0.09797326475381851
pred shape: torch.Size([491, 2])
attention_stack shape: torch.Size([3, 491, 491])
attention min: -10.234393119812012
attention max: 31.490766525268555
attention has NaN: False

year: 2019
window years: [2014, 2015, 2017, 2018, 2019]
best_epoch: 109
best_train_loss: 0.09993206212917964
pred shape: torch.Size([491, 2])
attention_stack shape: torch.Size([3, 491, 491])
attention min: -19.91347885131836
attention max: 43.185970306396484
attention has NaN: False

year: 2020
window years: [2015, 2017, 2018, 2019, 2020]
best_epoch: 228
best_train_loss: 0.08628447540104389
pred shape: torch.Size([491, 2])
attention_stack shape: torch.Size([3, 491, 491])
attention min: -30.430313110351562
attention max: 52.783477783203125
attention has NaN: False


### 5.2.5 Export `attention_weights_{year}.csv`

Sau khi lấy được `attention_stack` cho từng năm, bước này chuyển tensor attention thành bảng phẳng.

Mỗi dòng biểu diễn một quan hệ có hướng giữa hai công ty trong một năm và một attention head:

```text
src_company_id → dst_company_id
```

Output theo từng năm:

```text
attention_weights_2018.csv
attention_weights_2019.csv
attention_weights_2020.csv
```

Các file này là dữ liệu trung gian, dùng để tạo cạnh Top-K cho Attention Graph.


In [117]:
# Export attention_weights_2018.csv, attention_weights_2019.csv, attention_weights_2020.csv

import os
import pandas as pd
import numpy as np
import torch

ATTENTION_OUTPUT_DIR = "/content/project_outputs/attention_graph"
os.makedirs(ATTENTION_OUTPUT_DIR, exist_ok=True)

def export_attention_weights_for_year(test_year, attention_stack):
    test_year_idx = years.index(test_year)

    symbols_year = all_symbols[test_year_idx]
    labels_year = y_np[test_year_idx]

    assert len(symbols_year) == attention_stack.shape[1]
    assert len(labels_year) == attention_stack.shape[1]

    attention_np = attention_stack.detach().cpu().numpy()
    num_heads, n_src, n_dst = attention_np.shape

    rows = []

    for h in range(num_heads):
        for i in range(n_src):
            src_company_id = int(symbols_year[i])
            src_label = int(labels_year[i])

            for j in range(n_dst):
                dst_company_id = int(symbols_year[j])
                dst_label = int(labels_year[j])

                rows.append({
                    "year": test_year,
                    "head_id": h,
                    "src_index": i,
                    "dst_index": j,
                    "src_company_id": src_company_id,
                    "dst_company_id": dst_company_id,
                    "src_company_year_id": f"{src_company_id}_{test_year}",
                    "dst_company_year_id": f"{dst_company_id}_{test_year}",
                    "attention_weight": float(attention_np[h, i, j]),
                    "src_label": src_label,
                    "dst_label": dst_label,
                    "is_self_loop": int(i == j)
                })

    df = pd.DataFrame(rows)

    out_path = os.path.join(
        ATTENTION_OUTPUT_DIR,
        f"attention_weights_{test_year}.csv"
    )

    df.to_csv(out_path, index=False)

    print("\nĐã xuất:", out_path)
    print("shape:", df.shape)
    print("self_loop count:", df["is_self_loop"].sum())
    print("attention min:", df["attention_weight"].min())
    print("attention max:", df["attention_weight"].max())

    return out_path, df


attention_weight_files = {}

for year in TEST_YEARS:
    attention_stack = attention_results[year]["attention_stack"]
    out_path, _ = export_attention_weights_for_year(year, attention_stack)
    attention_weight_files[year] = out_path

print("\nAttention weight files:")
for year, path in attention_weight_files.items():
    print(year, "->", path)



Đã xuất: /content/project_outputs/attention_graph/attention_weights_2018.csv
shape: (723243, 12)
self_loop count: 1473
attention min: -10.234393119812012
attention max: 31.490766525268555

Đã xuất: /content/project_outputs/attention_graph/attention_weights_2019.csv
shape: (723243, 12)
self_loop count: 1473
attention min: -19.91347885131836
attention max: 43.185970306396484

Đã xuất: /content/project_outputs/attention_graph/attention_weights_2020.csv
shape: (723243, 12)
self_loop count: 1473
attention min: -30.430313110351562
attention max: 52.783477783203125

Attention weight files:
2018 -> /content/project_outputs/attention_graph/attention_weights_2018.csv
2019 -> /content/project_outputs/attention_graph/attention_weights_2019.csv
2020 -> /content/project_outputs/attention_graph/attention_weights_2020.csv


### 5.2.6 Tạo `attention_relation_edge.csv` bằng Top-K Attention

Từ các file `attention_weights_{year}.csv`, ta tạo cạnh Attention Graph chính thức cho 2018, 2019 và 2020.

Các bước xử lý cho từng năm:

1. Loại self-loop: `src_company_id != dst_company_id`.
2. Gộp nhiều attention heads bằng trung bình.
3. Khử hiệu ứng hub theo node đích bằng cách trừ đi attention trung bình của từng công ty đích.
4. Với mỗi công ty nguồn trong từng năm, chọn Top-K công ty đích có `attention_weight` cao nhất.
5. Sinh cạnh có hướng:

```text
src_company_year_id → dst_company_year_id
```

Output theo từng năm:

```text
attention_relation_edge_2018.csv
attention_relation_edge_2019.csv
attention_relation_edge_2020.csv
```

Output gộp để import NebulaGraph:

```text
attention_relation_edge.csv
```

Method sử dụng:

```text
mean_heads_dst_centered
```


In [118]:
# Tạo attention_relation_edge_{year}.csv và file gộp attention_relation_edge.csv

import os
import pandas as pd
import numpy as np

ATTENTION_OUTPUT_DIR = "/content/project_outputs/attention_graph"

TOP_K = 5
AGG_METHOD = "mean_heads_dst_centered"

def build_attention_edges_for_year(test_year):
    attention_weights_path = os.path.join(
        ATTENTION_OUTPUT_DIR,
        f"attention_weights_{test_year}.csv"
    )

    attention_edge_year_path = os.path.join(
        ATTENTION_OUTPUT_DIR,
        f"attention_relation_edge_{test_year}.csv"
    )

    assert os.path.exists(attention_weights_path), f"Không tìm thấy file: {attention_weights_path}"

    df_att = pd.read_csv(attention_weights_path)

    print("\n" + "=" * 80)
    print(f"BUILD ATTENTION EDGES FOR YEAR {test_year}")
    print("=" * 80)
    print("Loaded attention weights:", attention_weights_path)
    print("df_att shape:", df_att.shape)

    # 1. Loại self-loop
    df_no_self = df_att[df_att["is_self_loop"] == 0].copy()
    print("After removing self-loop:", df_no_self.shape)

    # 2. Gộp nhiều attention heads bằng trung bình
    group_cols = [
        "year",
        "src_index",
        "dst_index",
        "src_company_id",
        "dst_company_id",
        "src_company_year_id",
        "dst_company_year_id",
        "src_label",
        "dst_label"
    ]

    df_mean = (
        df_no_self
        .groupby(group_cols, as_index=False)
        .agg(attention_weight_raw=("attention_weight", "mean"))
    )

    # 3. Khử hub effect theo node đích
    df_mean["dst_mean_attention"] = (
        df_mean
        .groupby(["year", "dst_company_id"])["attention_weight_raw"]
        .transform("mean")
    )

    df_mean["attention_weight"] = (
        df_mean["attention_weight_raw"] - df_mean["dst_mean_attention"]
    )

    df_mean["method"] = AGG_METHOD

    # 4. Chọn Top-K theo từng src trong từng năm
    df_mean = df_mean.sort_values(
        by=["year", "src_company_id", "attention_weight"],
        ascending=[True, True, False]
    )

    df_mean["edge_rank"] = (
        df_mean
        .groupby(["year", "src_company_id"])
        .cumcount() + 1
    )

    df_topk = df_mean[df_mean["edge_rank"] <= TOP_K].copy()

    # 5. Chuẩn hóa schema edge để import NebulaGraph
    df_topk = df_topk.rename(columns={
        "src_company_year_id": "src",
        "dst_company_year_id": "dst"
    })

    df_topk["both_fraud"] = (
        (df_topk["src_label"] == 1) & (df_topk["dst_label"] == 1)
    ).astype(int)

    edge_cols = [
        "src",
        "dst",
        "year",
        "attention_weight",
        "attention_weight_raw",
        "dst_mean_attention",
        "src_company_id",
        "dst_company_id",
        "src_label",
        "dst_label",
        "both_fraud",
        "edge_rank",
        "method"
    ]

    df_edges = df_topk[edge_cols].copy()

    df_edges.to_csv(attention_edge_year_path, index=False)

    print("Đã xuất:", attention_edge_year_path)
    print("df_edges shape:", df_edges.shape)
    print("Expected edge count:", 491 * TOP_K)

    print("edge_count:", len(df_edges))
    print("unique src:", df_edges["src"].nunique())
    print("unique dst:", df_edges["dst"].nunique())
    print("same_label_ratio:", (df_edges["src_label"] == df_edges["dst_label"]).mean())
    print("fraud_fraud_ratio:", df_edges["both_fraud"].mean())
    print("attention_weight min:", df_edges["attention_weight"].min())
    print("attention_weight max:", df_edges["attention_weight"].max())

    return df_edges, attention_edge_year_path


all_edge_dfs = []
attention_edge_files = {}

for year in TEST_YEARS:
    df_edges_year, path_year = build_attention_edges_for_year(year)
    all_edge_dfs.append(df_edges_year)
    attention_edge_files[year] = path_year

df_attention_edges_all = pd.concat(all_edge_dfs, ignore_index=True)

attention_edge_all_path = os.path.join(
    ATTENTION_OUTPUT_DIR,
    "attention_relation_edge.csv"
)

df_attention_edges_all.to_csv(attention_edge_all_path, index=False)

print("\n" + "=" * 80)
print("Đã xuất file attention graph gộp để import NebulaGraph:")
print(attention_edge_all_path)
print("shape:", df_attention_edges_all.shape)
print("expected edge count:", 491 * TOP_K * len(TEST_YEARS))
print("years:", sorted(df_attention_edges_all["year"].unique()))
print("unique src:", df_attention_edges_all["src"].nunique())
print("unique dst:", df_attention_edges_all["dst"].nunique())
print("method:", df_attention_edges_all["method"].unique())



BUILD ATTENTION EDGES FOR YEAR 2018
Loaded attention weights: /content/project_outputs/attention_graph/attention_weights_2018.csv
df_att shape: (723243, 12)
After removing self-loop: (721770, 12)
Đã xuất: /content/project_outputs/attention_graph/attention_relation_edge_2018.csv
df_edges shape: (2455, 13)
Expected edge count: 2455
edge_count: 2455
unique src: 491
unique dst: 20
same_label_ratio: 0.6195519348268839
fraud_fraud_ratio: 0.06517311608961303
attention_weight min: -4.667515888749336
attention_weight max: 8.369004213566683

BUILD ATTENTION EDGES FOR YEAR 2019
Loaded attention weights: /content/project_outputs/attention_graph/attention_weights_2019.csv
df_att shape: (723243, 12)
After removing self-loop: (721770, 12)
Đã xuất: /content/project_outputs/attention_graph/attention_relation_edge_2019.csv
df_edges shape: (2455, 13)
Expected edge count: 2455
edge_count: 2455
unique src: 491
unique dst: 17
same_label_ratio: 0.6619144602851323
fraud_fraud_ratio: 0.04602851323828921
atten

In [119]:
# Kiểm tra các file attention relation edge cuối cùng

import os
import pandas as pd

ATTENTION_OUTPUT_DIR = "/content/project_outputs/attention_graph"

check_files = [
    "attention_relation_edge_2018.csv",
    "attention_relation_edge_2019.csv",
    "attention_relation_edge_2020.csv",
    "attention_relation_edge.csv",
]

for file_name in check_files:
    path = os.path.join(ATTENTION_OUTPUT_DIR, file_name)

    print("\n" + "=" * 80)
    print("File:", path)
    print("exists:", os.path.exists(path))

    if os.path.exists(path):
        df_check = pd.read_csv(path)
        print("Shape:", df_check.shape)
        print("Columns:", list(df_check.columns))
        print("years:", sorted(df_check["year"].unique()))
        print("unique src:", df_check["src"].nunique())
        print("unique dst:", df_check["dst"].nunique())
        print("edge_count:", len(df_check))
        print("method:", df_check["method"].unique())
        print(df_check.head())



File: /content/project_outputs/attention_graph/attention_relation_edge_2018.csv
exists: True
Shape: (2455, 13)
Columns: ['src', 'dst', 'year', 'attention_weight', 'attention_weight_raw', 'dst_mean_attention', 'src_company_id', 'dst_company_id', 'src_label', 'dst_label', 'both_fraud', 'edge_rank', 'method']
years: [np.int64(2018)]
unique src: 491
unique dst: 20
edge_count: 2455
method: ['mean_heads_dst_centered']
      src       dst  year  attention_weight  attention_weight_raw  \
0  1_2018  475_2018  2018         -4.354150             -7.153042   
1  1_2018  423_2018  2018         -4.410671             -6.290456   
2  1_2018   55_2018  2018         -4.411725             -6.444544   
3  1_2018  360_2018  2018         -4.413507             -6.512888   
4  1_2018  393_2018  2018         -4.428149             -6.251772   

   dst_mean_attention  src_company_id  dst_company_id  src_label  dst_label  \
0           -2.798892               1             475          0          0   
1         

### 5.3 Ghi `main_vera.py` — VerA (Code-faithful)

**Điểm khác biệt chính so với VerB:**

```python
# VerA — Early stopping: TRAIN LOSS thấp nhất (giữ nguyên tác giả)
if tr_loss < loss_min:
    loss_min = tr_loss
    best_epoch = epoch
    wait_count = 0

# VerA — Gradient: 1 lần sau toàn bộ epoch (giữ nguyên tác giả)
for i in train_seq:
    loss.backward()       # tích lũy gradient
optimizer.step()          # update 1 lần duy nhất
```

In [96]:
fixed_main_vera = r'''
import os, sys, json, random
import numpy as np
import torch
from torch import optim
import argparse
from model import *
from util4 import *

parser = argparse.ArgumentParser()
parser.add_argument("--data-path",   type=str,   default="/content/project_data/processed_flat.json")
parser.add_argument("--rnn-length",  type=int,   default=5)
parser.add_argument("--heads-att",   type=int,   default=3)
parser.add_argument("--hidn-att",    type=int,   default=32)
parser.add_argument("--hidn-rnn",    type=int,   default=64)
parser.add_argument("--crf-iter",    type=int,   default=5)
parser.add_argument("--max-epoch",   type=int,   default=800)
parser.add_argument("--wait-epoch",  type=int,   default=150)
parser.add_argument("--device",      type=str,   default="0")
parser.add_argument("--alpha",       type=float, default=0.5)
parser.add_argument("--batch-train", type=int,   default=4)
parser.add_argument("--clip",        type=float, default=1.0)
parser.add_argument("--seed",        type=int,   default=26541)
parser.add_argument("--lr",          type=float, default=1e-2)
parser.add_argument("--gamma",       type=int,   default=1)
parser.add_argument("--n-weight",    type=float, default=0.1)
parser.add_argument("--f-weight",    type=float, default=0.15)
parser.add_argument("--pool-type",   type=int,   default=3)
parser.add_argument("--test-year",   type=int,   default=2018)
parser.add_argument("--dropout",     type=float, default=0.0)
pool_dic = {1:"MAX", 2:"AVG", 3:"SUM"}

args        = parser.parse_args()
rnn_len     = args.rnn_length
MAX_EPOCH   = args.max_epoch
waits       = args.wait_epoch
test_year   = args.test_year
pool_type   = pool_dic[args.pool_type]
batch_train = args.batch_train
DEVICE = f"cuda:{args.device}" if torch.cuda.is_available() else "cpu"

torch.manual_seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)


def load_dataset(json_path):
    """Đọc processed_flat.json.
    JSON đã là [T, N, 209] nên KHÔNG dùng squeeze(axis=2).
    """
    with open(json_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    years = [b["year"] for b in payload["data"]]
    rows  = [b["rows"] for b in payload["data"]]

    data = torch.from_numpy(np.array(rows, dtype=np.float32)).float()
    x = data[:, :, :-1]                         # [T, N, 208]
    y = data[:, :, -1].unsqueeze(-1).long()    # [T, N, 1]  <-- FIX dtype

    print(f"Loaded: {len(years)} years | x={tuple(x.shape)}")
    print(f"Years : {years}")

    if test_year not in years:
        raise ValueError(f"test_year {test_year} not in {years}")

    test_idx = years.index(test_year)

    x_train = x[:test_idx]
    y_train = y[:test_idx]
    x_eval  = x[max(0, test_idx - rnn_len): test_idx + 1]
    y_eval  = y[max(0, test_idx - rnn_len): test_idx + 1]

    print(f"Train : {years[:test_idx]} ({x_train.shape[0]} years)")
    print(f"Eval  : {years[max(0, test_idx-rnn_len):test_idx+1]}")

    n_fraud  = int((y_train == 1).sum())
    n_benign = int((y_train == 0).sum())
    print(f"Train labels: fraud={n_fraud}, benign={n_benign}")

    return x_train, x_eval, y_train, y_eval


def do_train(model, x_train, y_train):
    """Train — giữ nguyên logic gốc tác giả:
    - optimizer.step() 1 lần sau toàn bộ vòng lặp
    - gradient tích lũy qua tất cả timestep trong epoch
    """
    model.train()
    seq_len   = len(x_train)
    train_seq = list(range(seq_len))[rnn_len:]

    if not train_seq:
        return 0., 0., 0., 0., 0.

    random.shuffle(train_seq)
    total_loss, cnt = 0., 0
    preds, trues    = [], []

    optimizer.zero_grad()

    for i in train_seq:
        y_i  = y_train[i].to(DEVICE).view(-1).long()   # <-- FIX dtype + shape
        out  = model(x_train[i - rnn_len + 1: i + 1].to(DEVICE))
        loss = criterion(out, y_i)
        loss.backward()

        total_loss += loss.item()
        cnt        += 1

        preds.append(out.detach().cpu().numpy())
        trues.append(y_train[i].cpu().numpy())

    torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
    optimizer.step()
    optimizer.zero_grad()

    acc, recall, pre, auc = train_metrics(trues, preds)
    return total_loss / max(cnt, 1), acc, recall, pre, auc


def do_eval(model, x_eval, y_eval):
    model.eval()
    seq = list(range(len(x_eval)))[rnn_len:]

    if not seq:
        return [0., 0., 0., 0.]

    preds, trues = [], []
    with torch.no_grad():
        for i in seq:
            out = model(x_eval[i - rnn_len + 1: i + 1].to(DEVICE))
            preds.append(out.detach().cpu().numpy())
            trues.append(y_eval[i].cpu().numpy())

    return metrics(trues, preds)


if __name__ == "__main__":
    SEP = "=" * 60
    print(f"\\nACRF-RNN VerA | test_year={test_year} | device={DEVICE}")
    print(SEP)

    x_train, x_eval, y_train, y_eval = load_dataset(args.data_path)

    num_company  = x_train.size(1)
    d_feature    = x_train.size(2)
    class_weight = torch.tensor([args.n_weight, args.f_weight]).to(DEVICE)
    criterion    = FocalLoss(gamma=args.gamma, weight=class_weight)

    model = AC_RNN(
        num_company=num_company,
        d_feature=d_feature,
        d_hidden=d_feature,
        hidn_rnn=args.hidn_rnn,
        heads_att=args.heads_att,
        hidn_att=args.hidn_att,
        crf_iters=args.crf_iter,
        alpha=args.alpha,
        pool_type=pool_type
    ).to(DEVICE)

    model.drop_out = args.dropout
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    print(f"Model params: {sum(p.numel() for p in model.parameters()):,}\\n")

    loss_min    = float("inf")
    best_epoch  = 0
    best_m      = None
    wait_count  = 0
    epoch       = 0

    HDR = f"{'Ep':>4} | {'Loss':>7} | {'TrAcc':>6} | {'EvAcc':>6} | {'Rcl_m':>6} | {'KS':>6} | {'GM':>6} |"
    print(HDR)
    print("-" * len(HDR))

    import time
    t_start = time.time()

    while epoch < MAX_EPOCH:
        tr_loss, tr_acc, tr_rcl, _, _ = do_train(model, x_train, y_train)
        ev = do_eval(model, x_eval, y_eval)

        is_best = ""
        if tr_loss < loss_min:
            loss_min   = tr_loss
            best_epoch = epoch
            best_m     = ev
            wait_count = 0
            is_best    = "best"
        else:
            wait_count += 1

        print(
            f"{epoch:>4} | {tr_loss:>7.4f} | {tr_acc:>6.4f} | "
            f"{ev[0]:>6.4f} | {ev[1]:>6.4f} | {ev[2]:>6.4f} | {ev[3]:>6.4f} | {is_best}"
        )

        if (epoch + 1) % 10 == 0 or is_best == "best":
            elapsed = time.time() - t_start
            pct = (epoch + 1) / MAX_EPOCH * 100
            eta = elapsed / (epoch + 1) * (MAX_EPOCH - epoch - 1)
            print(
                f"  --> [{pct:5.1f}%] ep {epoch+1}/{MAX_EPOCH} | "
                f"elapsed {elapsed:.0f}s | ETA {eta:.0f}s | "
                f"best loss {loss_min:.4f} @ ep{best_epoch} | "
                f"wait {wait_count}/{waits}"
            )

        if wait_count >= waits:
            print(f"\\nEarly stop @ epoch {epoch}  (best_epoch={best_epoch})")
            break

        epoch += 1

    print(f"\\n{SEP}")
    print(f"KET QUA VerA -- test_year={test_year}")
    print(SEP)

    if best_m:
        paper = {
            2018: (77.60, 70.79, 41.58, 69.37),
            2019: (80.45, 75.44, 50.88, 74.89),
            2020: (79.43, 71.96, 43.92, 71.04),
        }
        p = paper.get(test_year, (0, 0, 0, 0))

        print(f"Best epoch (by train_loss): {best_epoch}")
        print(f"Best train_loss           : {loss_min:.4f}")
        print(f"{'Metric':<12} {'Paper':>7} {'VerA':>7} {'Diff':>7}")
        print("-" * 40)

        for name, pv, ov in zip(
                ["Accuracy", "Recall_m", "KS", "G-mean"],
                p,
                [m * 100 for m in best_m]):
            d = ov - pv
            print(f"{name:<12} {pv:>7.2f} {ov:>7.2f} {'+' if d >= 0 else ''}{d:>6.2f}")
'''

with open("/content/ACRF-RNN/Code/main_vera.py", "w", encoding="utf-8") as f:
    f.write(fixed_main_vera)

print("Đã ghi /content/ACRF-RNN/Code/main_vera.py")
!wc -l /content/ACRF-RNN/Code/main_vera.py

Đã ghi /content/ACRF-RNN/Code/main_vera.py
240 /content/ACRF-RNN/Code/main_vera.py


### 5.4 Ghi `main.py` — VerB (Paper-faithful)

**Điểm khác biệt chính so với VerA:**

```python
# VerB — Early stopping: EVAL KS cao nhất (theo bài báo Section 5.3)
if ev[2] > best_ks:
    best_ks = ev[2]
    best_epoch = epoch
    wait_count = 0

# VerB — Gradient: mỗi batch_train=4 bước (theo bài báo Section 5.4)
for i in train_seq:
    loss.backward()
    if cnt % batch_train == batch_train - 1:
        optimizer.step()  # update mỗi 4 bước
        optimizer.zero_grad()
```

In [97]:
# main.py — Fix 1, 3, 4, 5, 6 + đọc JSON từ Spark pipeline
fixed_main_verb = r'''
import os, sys, json, random
import numpy as np
import torch
from torch import optim
import argparse
from model import *
from util4 import *

parser = argparse.ArgumentParser()
parser.add_argument("--data-path",   type=str,   default="/content/project_data/processed_flat.json")
parser.add_argument("--rnn-length",  type=int,   default=5)
parser.add_argument("--heads-att",   type=int,   default=3)
parser.add_argument("--hidn-att",    type=int,   default=32)
parser.add_argument("--hidn-rnn",    type=int,   default=64)
parser.add_argument("--crf-iter",    type=int,   default=5)
parser.add_argument("--max-epoch",   type=int,   default=800)
parser.add_argument("--wait-epoch",  type=int,   default=150)
parser.add_argument("--device",      type=str,   default="0")
parser.add_argument("--alpha",       type=float, default=0.5)
parser.add_argument("--batch-train", type=int,   default=4)
parser.add_argument("--clip",        type=float, default=1.0)
parser.add_argument("--seed",        type=int,   default=26541)
parser.add_argument("--lr",          type=float, default=1e-2)
parser.add_argument("--gamma",       type=int,   default=1)
parser.add_argument("--n-weight",    type=float, default=0.1)
parser.add_argument("--f-weight",    type=float, default=0.15)
parser.add_argument("--pool-type",   type=int,   default=3)
parser.add_argument("--test-year",   type=int,   default=2018)
parser.add_argument("--dropout",     type=float, default=0.0)
pool_dic = {1:"MAX", 2:"AVG", 3:"SUM"}

args        = parser.parse_args()
rnn_len     = args.rnn_length
MAX_EPOCH   = args.max_epoch
waits       = args.wait_epoch
test_year   = args.test_year
pool_type   = pool_dic[args.pool_type]
batch_train = args.batch_train
DEVICE = f"cuda:{args.device}" if torch.cuda.is_available() else "cpu"

torch.manual_seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)


def load_dataset(json_path):
    """Doc processed_flat.json tu Spark pipeline.
    FIX 1: bo np.squeeze(axis=2) — JSON da la [T,N,209], khong co chieu thua.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    years = [b["year"] for b in payload["data"]]
    rows  = [b["rows"] for b in payload["data"]]
    data  = torch.from_numpy(np.array(rows, dtype=np.float32)).float()
    # data shape: [T, N, 209]
    # (CSV gốc có shape [T,N,1,209] vì groupby tạo thêm 1 chiều thừa;
    #  JSON đã flat nên không cần squeeze(axis=2) nua)
    y = data[:, :, -1].unsqueeze(-1).long()   # [T, N, 1]
    x = data[:, :, :-1]                        # [T, N, 208]
    print(f"Loaded: {len(years)} years | x={tuple(x.shape)} | y={tuple(y.shape)}")
    print(f"Years : {years}")
    if test_year not in years:
        raise ValueError(f"test_year {test_year} not in {years}")
    test_idx   = years.index(test_year)
    x_train    = x[:test_idx]
    y_train    = y[:test_idx]
    eval_start = max(0, test_idx - rnn_len)
    x_eval     = x[eval_start:test_idx + 1]
    y_eval     = y[eval_start:test_idx + 1]
    print(f"Train : {years[:test_idx]}  ({x_train.shape[0]} years)")
    print(f"Eval  : {years[eval_start:test_idx+1]}")
    n_fraud  = int((y_train == 1).sum())
    n_benign = int((y_train == 0).sum())
    print(f"Train labels: fraud={n_fraud}, benign={n_benign}")
    return x_train, x_eval, y_train, y_eval


def do_train(model, x_train, y_train):
    model.train()
    seq_len   = len(x_train)
    train_seq = list(range(seq_len))[rnn_len:]
    if not train_seq:
        return 0., 0., 0., 0., 0.
    random.shuffle(train_seq)   # Fix 6: bỏ tham số random= (Python >= 3.9)
    total_loss, cnt = 0., 0
    preds, trues    = [], []
    optimizer.zero_grad()
    for i in train_seq:
        y_i  = torch.squeeze(y_train[i], 1).to(DEVICE)
        out  = model(x_train[i - rnn_len + 1: i + 1].to(DEVICE))
        loss = criterion(out, y_i)
        loss.backward()
        total_loss += loss.item()
        cnt        += 1
        if cnt % batch_train == batch_train - 1:
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
            optimizer.step()
            optimizer.zero_grad()
        preds.append(out.detach().cpu().numpy())
        trues.append(y_train[i].cpu().numpy())
    if cnt % batch_train != batch_train - 1:
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
        optimizer.step()
        optimizer.zero_grad()
    acc, recall, pre, auc = train_metrics(trues, preds)
    return total_loss / max(cnt, 1), acc, recall, pre, auc


def do_eval(model, x_eval, y_eval):
    """Eval — KS dung ks_2samp tu util4.py (dung dinh nghia bai bao).
    FIX 3: bo plt_output chua dinh nghia trong main.py goc.
    """
    model.eval()
    seq = list(range(len(x_eval)))[rnn_len:]
    if not seq:
        return [0., 0., 0., 0.]
    preds, trues = [], []
    with torch.no_grad():
        for i in seq:
            out = model(x_eval[i - rnn_len + 1: i + 1].to(DEVICE))
            preds.append(out.detach().cpu().numpy())
            trues.append(y_eval[i].cpu().numpy())
    return metrics(trues, preds)   # [acc, recall_macro, KS(ks_2samp), G-mean]


if __name__ == "__main__":
    SEP = "=" * 60
    print(f"\nACRF-RNN | test_year={test_year} | device={DEVICE}")
    print(SEP)

    x_train, x_eval, y_train, y_eval = load_dataset(args.data_path)

    num_company  = x_train.size(1)
    d_feature    = x_train.size(2)
    class_weight = torch.tensor([args.n_weight, args.f_weight]).to(DEVICE)
    criterion    = FocalLoss(gamma=args.gamma, weight=class_weight)

    model = AC_RNN(
        num_company=num_company,
        d_feature=d_feature,
        d_hidden=d_feature,
        hidn_rnn=args.hidn_rnn,
        heads_att=args.heads_att,
        hidn_att=args.hidn_att,
        crf_iters=args.crf_iter,
        alpha=args.alpha,
        pool_type=pool_type
    ).to(DEVICE)
    model.drop_out = args.dropout
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    print(f"Model params: {sum(p.numel() for p in model.parameters()):,}\n")

    best_ks, best_epoch, best_m = -1., 0, None
    wait_count, epoch = 0, 0

    HDR = f"{'Ep':>4} | {'Loss':>7} | {'TrAcc':>6} | {'EvAcc':>6} | {'Rcl_m':>6} | {'KS':>6} | {'GM':>6} |"
    print(HDR)
    print("-" * len(HDR))

    from tqdm import tqdm
    pbar = tqdm(total=MAX_EPOCH, desc=f"test_year={test_year}",
                unit="ep", dynamic_ncols=True,
                bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} ep [{elapsed}<{remaining}]")

    while epoch < MAX_EPOCH:
        tr_loss, tr_acc, tr_rcl, _, _ = do_train(model, x_train, y_train)
        ev = do_eval(model, x_eval, y_eval)
        is_best = ""
        if ev[2] > best_ks:
            best_ks, best_epoch, best_m = ev[2], epoch, ev
            wait_count = 0
            is_best    = "best"
        else:
            wait_count += 1
        print(
            f"{epoch:>4} | {tr_loss:>7.4f} | {tr_acc:>6.4f} | "
            f"{ev[0]:>6.4f} | {ev[1]:>6.4f} | {ev[2]:>6.4f} | {ev[3]:>6.4f} | {is_best}"
        )
        # Cap nhat tqdm: hien thi epoch hien tai, KS tot nhat, patience con lai
        pbar.update(1)
        pbar.set_postfix({
            "loss": f"{tr_loss:.4f}",
            "KS": f"{ev[2]*100:.2f}%",
            "best_KS": f"{best_ks*100:.2f}%@ep{best_epoch}",
            "wait": f"{wait_count}/{waits}",
        }, refresh=True)
        if wait_count >= waits:
            pbar.set_description(f"Early stop ep{epoch} | best ep{best_epoch}")
            pbar.close()
            print(f"\nEarly stop @ epoch {epoch}  (best_epoch={best_epoch})")
            break
        epoch += 1
    else:
        pbar.close()

    print(f"\n{SEP}")
    print(f"KET QUA -- test_year={test_year}")
    print(SEP)
    if best_m:
        paper = {
            2018: (77.60, 70.79, 41.58, 69.37),
            2019: (80.45, 75.44, 50.88, 74.89),
            2020: (79.43, 71.96, 43.92, 71.04),
        }
        p = paper.get(test_year, (0, 0, 0, 0))
        print(f"Best epoch : {best_epoch}")
        print(f"{'Metric':<12} {'Paper':>7} {'VerB':>7} {'Diff':>7}")
        print("-" * 40)
        for name, pv, ov in zip(
                ["Accuracy", "Recall_m", "KS", "G-mean"],
                p,
                [m * 100 for m in best_m]):
            d = ov - pv
            print(f"{name:<12} {pv:>7.2f} {ov:>7.2f} {'+' if d >= 0 else ''}{d:>6.2f}")
'''

with open("/content/ACRF-RNN/Code/main.py", "w", encoding="utf-8") as f:
    f.write(fixed_main_verb)
print("Đã ghi /content/ACRF-RNN/Code/main.py")
!wc -l /content/ACRF-RNN/Code/main.py

Đã ghi /content/ACRF-RNN/Code/main.py
217 /content/ACRF-RNN/Code/main.py


## Giai đoạn 6 — Thực nghiệm

### 6.1 Smoke test (5 epoch) — kiểm tra pipeline không lỗi

In [98]:
%cd /content/ACRF-RNN/Code
!python main.py \
    --data-path /content/project_data/processed_flat.json \
    --test-year 2018 --max-epoch 5 --wait-epoch 3 --device 0 \
    2>&1 | head -30

/content/ACRF-RNN/Code

ACRF-RNN | test_year=2018 | device=cuda:0
Loaded: 10 years | x=(10, 491, 208) | y=(10, 491, 1)
Years : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020]
Train : [2010, 2011, 2012, 2013, 2014, 2015, 2017]  (7 years)
Eval  : [2012, 2013, 2014, 2015, 2017, 2018]
Train labels: fraud=860, benign=2577
Model params: 972,937

  Ep |    Loss |  TrAcc |  EvAcc |  Rcl_m |     KS |     GM |
-------------------------------------------------------------
Early stop ep3 | best ep0:  80%|████████  | 4/5 ep [00:00<00:00]
   0 |  0.4153 | 0.5580 | 0.4725 | 0.4469 | 0.1063 | 0.4437 | best
   1 |  0.4404 | 0.5204 | 0.5214 | 0.4926 | 0.0147 | 0.4890 | 
   2 |  0.4085 | 0.4939 | 0.5356 | 0.5048 | 0.0096 | 0.5008 | 
   3 |  0.3919 | 0.5458 | 0.5743 | 0.5335 | 0.0670 | 0.5267 | 

Early stop @ epoch 3  (best_epoch=0)

KET QUA -- test_year=2018
Best epoch : 0
Metric         Paper    VerB    Diff
----------------------------------------
Accuracy       77.60   47.25 -30.35
Recall

### 6.2–6.4 VerA — Code-faithful

Chạy `main_vera.py` với early stopping theo **train_loss**.

#### 6.2 VerA — Test year 2018

In [99]:
%cd /content/ACRF-RNN/Code
!ls -lah

/content/ACRF-RNN/Code
total 56K
drwxr-xr-x 4 root root 4.0K Jun  2 09:05 .
drwxr-xr-x 5 root root 4.0K Jun  2 09:02 ..
-rw-r--r-- 1 root root 5.2K Jun  2 09:02 layer.py
-rw-r--r-- 1 root root 8.3K Jun  2 09:05 main.py
-rw-r--r-- 1 root root 7.8K Jun  2 09:05 main_vera.py
-rw-r--r-- 1 root root 3.1K Jun  2 09:02 model_attention.py
-rw-r--r-- 1 root root 2.2K Jun  2 09:02 model.py
drwxr-xr-x 2 root root 4.0K Jun  2 09:05 __pycache__
drwxr-xr-x 2 root root 4.0K Jun  2 09:02 savemodel
-rw-r--r-- 1 root root 2.4K Jun  2 09:02 util4.py


In [100]:
%cd /content/ACRF-RNN/Code
!python main_vera.py \
    --data-path /content/project_data/processed_flat.json \
    --test-year 2018 --max-epoch 800 --wait-epoch 150 --device 0 \
    2>&1 | tee /content/project_outputs/run_vera_2018.log

/content/ACRF-RNN/Code
\nACRF-RNN VerA | test_year=2018 | device=cuda:0
Loaded: 10 years | x=(10, 491, 208)
Years : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020]
Train : [2010, 2011, 2012, 2013, 2014, 2015, 2017] (7 years)
Eval  : [2012, 2013, 2014, 2015, 2017, 2018]
Train labels: fraud=860, benign=2577
Model params: 972,937\n
  Ep |    Loss |  TrAcc |  EvAcc |  Rcl_m |     KS |     GM |
-------------------------------------------------------------
   0 |  0.4153 | 0.5580 | 0.4725 | 0.4469 | 0.1063 | 0.4437 | best
  --> [  0.1%] ep 1/800 | elapsed 0s | ETA 355s | best loss 0.4153 @ ep0 | wait 0/150
   1 |  0.4404 | 0.5204 | 0.5214 | 0.4926 | 0.0147 | 0.4890 | 
   2 |  0.4085 | 0.4939 | 0.5356 | 0.5048 | 0.0096 | 0.5008 | best
  --> [  0.4%] ep 3/800 | elapsed 1s | ETA 145s | best loss 0.4085 @ ep2 | wait 0/150
   3 |  0.3919 | 0.5458 | 0.5743 | 0.5335 | 0.0670 | 0.5267 | best
  --> [  0.5%] ep 4/800 | elapsed 1s | ETA 118s | best loss 0.3919 @ ep3 | wait 0/150
   4 |  0.

#### 6.3 VerA — Test year 2019

In [101]:
%cd /content/ACRF-RNN/Code
!python main_vera.py \
    --data-path /content/project_data/processed_flat.json \
    --test-year 2019 --max-epoch 800 --wait-epoch 150 --device 0 \
    2>&1 | tee /content/project_outputs/run_vera_2019.log

/content/ACRF-RNN/Code
\nACRF-RNN VerA | test_year=2019 | device=cuda:0
Loaded: 10 years | x=(10, 491, 208)
Years : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020]
Train : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018] (8 years)
Eval  : [2013, 2014, 2015, 2017, 2018, 2019]
Train labels: fraud=987, benign=2941
Model params: 972,937\n
  Ep |    Loss |  TrAcc |  EvAcc |  Rcl_m |     KS |     GM |
-------------------------------------------------------------
   0 |  0.4150 | 0.5553 | 0.5621 | 0.5400 | 0.0801 | 0.5385 | best
  --> [  0.1%] ep 1/800 | elapsed 0s | ETA 361s | best loss 0.4150 @ ep0 | wait 0/150
   1 |  0.3941 | 0.4956 | 0.5153 | 0.5034 | 0.0068 | 0.5029 | best
  --> [  0.2%] ep 2/800 | elapsed 1s | ETA 205s | best loss 0.3941 @ ep1 | wait 0/150
   2 |  0.3786 | 0.5485 | 0.6191 | 0.5929 | 0.1859 | 0.5910 | best
  --> [  0.4%] ep 3/800 | elapsed 1s | ETA 152s | best loss 0.3786 @ ep2 | wait 0/150
   3 |  0.3260 | 0.6144 | 0.5682 | 0.5860 | 0.1720 | 0.5851 | best


#### 6.4 VerA — Test year 2020

In [102]:
%cd /content/ACRF-RNN/Code
!python main_vera.py \
    --data-path /content/project_data/processed_flat.json \
    --test-year 2020 --max-epoch 800 --wait-epoch 150 --device 0 \
    2>&1 | tee /content/project_outputs/run_vera_2020.log

/content/ACRF-RNN/Code
\nACRF-RNN VerA | test_year=2020 | device=cuda:0
Loaded: 10 years | x=(10, 491, 208)
Years : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020]
Train : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019] (9 years)
Eval  : [2014, 2015, 2017, 2018, 2019, 2020]
Train labels: fraud=1097, benign=3322
Model params: 972,937\n
  Ep |    Loss |  TrAcc |  EvAcc |  Rcl_m |     KS |     GM |
-------------------------------------------------------------
   0 |  0.4064 | 0.5738 | 0.5764 | 0.4868 | 0.0265 | 0.4668 | best
  --> [  0.1%] ep 1/800 | elapsed 1s | ETA 535s | best loss 0.4064 @ ep0 | wait 0/150
   1 |  0.3938 | 0.4980 | 0.5519 | 0.5406 | 0.0813 | 0.5404 | best
  --> [  0.2%] ep 2/800 | elapsed 1s | ETA 321s | best loss 0.3938 @ ep1 | wait 0/150
   2 |  0.4133 | 0.5346 | 0.5764 | 0.5692 | 0.1384 | 0.5691 | 
   3 |  0.3820 | 0.4939 | 0.5621 | 0.5560 | 0.1119 | 0.5559 | best
  --> [  0.5%] ep 4/800 | elapsed 1s | ETA 209s | best loss 0.3820 @ ep3 | wait 0/1

### 6.5–6.7 VerB — Paper-faithful

Chạy `main.py` với early stopping theo **eval KS**.

#### 6.5 VerB — Test year 2018

In [103]:
%cd /content/ACRF-RNN/Code
!python main.py \
    --data-path /content/project_data/processed_flat.json \
    --test-year 2018 --max-epoch 800 --wait-epoch 150 --device 0 \
    2>&1 | tee /content/project_outputs/run_2018.log

/content/ACRF-RNN/Code

ACRF-RNN | test_year=2018 | device=cuda:0
Loaded: 10 years | x=(10, 491, 208) | y=(10, 491, 1)
Years : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020]
Train : [2010, 2011, 2012, 2013, 2014, 2015, 2017]  (7 years)
Eval  : [2012, 2013, 2014, 2015, 2017, 2018]
Train labels: fraud=860, benign=2577
Model params: 972,937

  Ep |    Loss |  TrAcc |  EvAcc |  Rcl_m |     KS |     GM |
-------------------------------------------------------------
test_year=2018:  16%|█▌        | 128/800 ep [00:07<00:31]   0 |  0.4153 | 0.5580 | 0.4725 | 0.4469 | 0.1063 | 0.4437 | best
   1 |  0.4404 | 0.5204 | 0.5214 | 0.4926 | 0.0147 | 0.4890 | 
   2 |  0.4085 | 0.4939 | 0.5356 | 0.5048 | 0.0096 | 0.5008 | 
   3 |  0.3919 | 0.5458 | 0.5743 | 0.5335 | 0.0670 | 0.5267 | 
   4 |  0.2982 | 0.6527 | 0.5927 | 0.5458 | 0.0917 | 0.5371 | 
   5 |  0.2488 | 0.7251 | 0.6395 | 0.5902 | 0.1805 | 0.5814 | best
   6 |  0.2280 | 0.7648 | 0.6864 | 0.6449 | 0.2898 | 0.6392 | best
   7 |  0.4

#### 6.6 VerB — Test year 2019

In [104]:
%cd /content/ACRF-RNN/Code
!python main.py \
    --data-path /content/project_data/processed_flat.json \
    --test-year 2019 --max-epoch 800 --wait-epoch 150 --device 0 \
    2>&1 | tee /content/project_outputs/run_2019.log

/content/ACRF-RNN/Code

ACRF-RNN | test_year=2019 | device=cuda:0
Loaded: 10 years | x=(10, 491, 208) | y=(10, 491, 1)
Years : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020]
Train : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018]  (8 years)
Eval  : [2013, 2014, 2015, 2017, 2018, 2019]
Train labels: fraud=987, benign=2941
Model params: 972,937

  Ep |    Loss |  TrAcc |  EvAcc |  Rcl_m |     KS |     GM |
-------------------------------------------------------------
test_year=2019:  16%|█▌        | 129/800 ep [00:09<00:43]   0 |  0.4150 | 0.5553 | 0.5621 | 0.5400 | 0.0801 | 0.5385 | best
   1 |  0.3941 | 0.4956 | 0.5153 | 0.5034 | 0.0068 | 0.5029 | 
   2 |  0.3786 | 0.5485 | 0.6191 | 0.5929 | 0.1859 | 0.5910 | best
   3 |  0.3260 | 0.6144 | 0.5682 | 0.5860 | 0.1720 | 0.5851 | 
   4 |  0.2711 | 0.6979 | 0.7128 | 0.6792 | 0.3583 | 0.6764 | best
   5 |  0.2395 | 0.7502 | 0.7413 | 0.6975 | 0.3951 | 0.6930 | best
   6 |  0.2115 | 0.8018 | 0.7597 | 0.6964 | 0.3928 | 0.6869 | 
 

#### 6.7 VerB — Test year 2020

In [105]:
%cd /content/ACRF-RNN/Code
!python main.py \
    --data-path /content/project_data/processed_flat.json \
    --test-year 2020 --max-epoch 800 --wait-epoch 150 --device 0 \
    2>&1 | tee /content/project_outputs/run_2020.log

/content/ACRF-RNN/Code

ACRF-RNN | test_year=2020 | device=cuda:0
Loaded: 10 years | x=(10, 491, 208) | y=(10, 491, 1)
Years : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020]
Train : [2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019]  (9 years)
Eval  : [2014, 2015, 2017, 2018, 2019, 2020]
Train labels: fraud=1097, benign=3322
Model params: 972,937

  Ep |    Loss |  TrAcc |  EvAcc |  Rcl_m |     KS |     GM |
-------------------------------------------------------------
test_year=2020:  16%|█▌        | 129/800 ep [00:11<00:55]   0 |  0.4190 | 0.5570 | 0.5886 | 0.5537 | 0.1074 | 0.5511 | best
   1 |  0.3874 | 0.4893 | 0.5519 | 0.5315 | 0.0630 | 0.5305 | 
   2 |  0.3752 | 0.5703 | 0.6069 | 0.5557 | 0.1113 | 0.5500 | best
   3 |  0.3077 | 0.6436 | 0.7556 | 0.6824 | 0.3648 | 0.6731 | best
   4 |  0.2626 | 0.7449 | 0.7128 | 0.6199 | 0.2397 | 0.6031 | 
   5 |  0.2900 | 0.7179 | 0.7902 | 0.6897 | 0.3793 | 0.6721 | best
   6 |  0.3055 | 0.6650 | 0.7617 | 0.6815 | 0.3631 | 0.67

## Giai đoạn 7 — Tổng hợp và so sánh kết quả

### 7.1 Bảng VerA — best theo train_loss (tiêu chí tác giả)



In [106]:
import re, pandas as pd, os

pat = re.compile(
    r"\s*(\d+)\s*\|\s*([\d.]+)\s*\|\s*([\d.]+)\s*\|"
    r"\s*([\d.]+)\s*\|\s*([\d.]+)\s*\|\s*([\d.]+)\s*\|\s*([\d.]+)"
)


def parse_log_vera(log_path, test_year):
    """VerA: best epoch = epoch có train_loss thấp nhất (theo tác giả)."""
    epoch_rows, best_loss_row, best_ks_row, last_ep = [], {"loss": float("inf")}, {"raw": -1}, None
    with open(log_path) as f:
        for line in f:
            m = pat.match(line)
            if m:
                ep, loss, tr_acc, ev_acc, rcl, ks, gm = [float(x) for x in m.groups()]
                epoch_rows.append(dict(
                    test_year=test_year, epoch=int(ep),
                    loss=round(loss,4), train_acc=round(tr_acc*100,2),
                    eval_acc=round(ev_acc*100,2), eval_recall_macro=round(rcl*100,2),
                    eval_ks=round(ks*100,2), eval_gm=round(gm*100,2),
                ))
                if loss < best_loss_row["loss"]:
                    best_loss_row = {"loss": round(loss,4), "epoch": int(ep),
                        "eval_acc": round(ev_acc*100,2), "eval_recall_macro": round(rcl*100,2),
                        "eval_ks":  round(ks*100,2),    "eval_gm": round(gm*100,2)}
                if ks > best_ks_row["raw"]:
                    best_ks_row = {"raw": ks, "epoch": int(ep), "val": round(ks*100,2)}
                last_ep = {"epoch": int(ep), "eval_acc": round(ev_acc*100,2),
                    "eval_recall_macro": round(rcl*100,2),
                    "eval_ks": round(ks*100,2), "eval_gm": round(gm*100,2)}
    return epoch_rows, {
        "test_year": test_year,
        "best_epoch_by_loss":      best_loss_row.get("epoch"),
        "best_train_loss":         best_loss_row.get("loss"),
        "best_eval_acc":           best_loss_row.get("eval_acc"),
        "best_eval_recall_macro":  best_loss_row.get("eval_recall_macro"),
        "best_eval_ks":            best_loss_row.get("eval_ks"),
        "best_eval_gm":            best_loss_row.get("eval_gm"),
        "best_epoch_by_ks":        best_ks_row.get("epoch"),
        "ref_best_ks":             best_ks_row.get("val"),
        "final_eval_acc":          last_ep["eval_acc"]          if last_ep else None,
        "final_eval_recall_macro": last_ep["eval_recall_macro"] if last_ep else None,
        "final_eval_ks":           last_ep["eval_ks"]           if last_ep else None,
        "final_eval_gm":           last_ep["eval_gm"]           if last_ep else None,
    }


vera_logs = {2018: "/content/project_outputs/run_vera_2018.log",
             2019: "/content/project_outputs/run_vera_2019.log",
             2020: "/content/project_outputs/run_vera_2020.log"}

vera_epochs, vera_rows = [], []
for yr, path in vera_logs.items():
    if not os.path.exists(path):
        print(f"Chưa có {path}"); continue
    rows, summary = parse_log_vera(path, yr)
    vera_epochs.extend(rows); vera_rows.append(summary)

df_vera_epochs  = pd.DataFrame(vera_epochs)
df_vera_summary = pd.DataFrame(vera_rows)
print(f"VerA epoch rows: {len(df_vera_epochs)}")
df_vera_summary

VerA epoch rows: 812


,test_year,best_epoch_by_loss,best_train_loss,best_eval_acc,best_eval_recall_macro,best_eval_ks,best_eval_gm,best_epoch_by_ks,ref_best_ks,final_eval_acc,final_eval_recall_macro,final_eval_ks,final_eval_gm
0,2018,22,0.0980,75.36,69.54,39.07,68.48,46,42.96,71.49,64.62,29.24,63.03
1,2019,109,0.0999,77.39,73.79,47.59,73.51,148,49.27,55.80,55.68,11.36,55.68
2,2020,227,0.0863,81.47,73.20,46.39,72.08,210,47.98,80.24,72.45,44.91,71.46


### Nhận xét 7.1 — Kết quả VerA

Kết quả VerA cho thấy mô hình đạt mức tái hiện khá tốt so với bài báo gốc ở cả ba năm kiểm thử. Ở năm 2018, VerA đạt KS = 39.07%, Recallm = 69.54% và G-mean = 68.48%, chỉ thấp hơn bài báo một khoảng nhỏ, cho thấy chất lượng mô hình đã khá sát với kết quả công bố. Ở năm 2019, VerA tiếp tục duy trì mức kết quả tốt với KS = 47.59%, Recallm = 73.79% và G-mean = 73.51%, vẫn nằm rất gần bài báo. Đặc biệt, ở năm 2020, VerA đạt KS = 46.39%, Recallm = 73.20% và G-mean = 72.08%, đều cao hơn bài báo gốc, cho thấy thiết lập này hoạt động rất tốt ở năm kiểm thử cuối.

Một điểm đáng chú ý là chỉ số `final_eval_*` của VerA ở một số năm, đặc biệt năm 2019, giảm khá mạnh so với `best_*`. Điều này cho thấy nếu lấy epoch cuối cùng thì kết quả sẽ kém hơn đáng kể, và việc chọn best epoch theo train loss là cần thiết để tránh hiện tượng mô hình suy giảm hiệu quả ở giai đoạn cuối huấn luyện.

Nhìn chung, VerA có ưu điểm là ổn định, bám sát bài báo ở cả ba năm và đặc biệt mạnh ở năm 2020.

### 7.2 Bảng VerB — best theo eval KS (tiêu chí bài báo)

In [107]:
def parse_log_verb(log_path, test_year):
    """VerB: best epoch = epoch có eval KS cao nhất (theo bài báo Section 5.3)."""
    epoch_rows  = []
    best_ks_row = {"raw": -1}
    last_ep     = None
    with open(log_path) as f:
        for line in f:
            m = pat.match(line)
            if m:
                ep, loss, tr_acc, ev_acc, rcl, ks, gm = [float(x) for x in m.groups()]
                epoch_rows.append(dict(
                    test_year=test_year, epoch=int(ep),
                    loss=round(loss,4), train_acc=round(tr_acc*100,2),
                    eval_acc=round(ev_acc*100,2), eval_recall_macro=round(rcl*100,2),
                    eval_ks=round(ks*100,2), eval_gm=round(gm*100,2),
                ))
                if ks > best_ks_row["raw"]:
                    best_ks_row = {"raw": ks, "epoch": int(ep),
                        "eval_acc": round(ev_acc*100,2), "eval_recall_macro": round(rcl*100,2),
                        "eval_ks":  round(ks*100,2),    "eval_gm": round(gm*100,2)}
                last_ep = {"epoch": int(ep), "eval_acc": round(ev_acc*100,2),
                    "eval_recall_macro": round(rcl*100,2),
                    "eval_ks": round(ks*100,2), "eval_gm": round(gm*100,2)}
    return epoch_rows, {
        "test_year":               test_year,
        "best_epoch_by_ks":        best_ks_row.get("epoch"),
        "best_eval_acc":           best_ks_row.get("eval_acc"),
        "best_eval_recall_macro":  best_ks_row.get("eval_recall_macro"),
        "best_eval_ks":            best_ks_row.get("eval_ks"),
        "best_eval_gm":            best_ks_row.get("eval_gm"),
        "final_eval_acc":          last_ep["eval_acc"]          if last_ep else None,
        "final_eval_recall_macro": last_ep["eval_recall_macro"] if last_ep else None,
        "final_eval_ks":           last_ep["eval_ks"]           if last_ep else None,
        "final_eval_gm":           last_ep["eval_gm"]           if last_ep else None,
    }


verb_logs = {2018: "/content/project_outputs/run_2018.log",
             2019: "/content/project_outputs/run_2019.log",
             2020: "/content/project_outputs/run_2020.log"}

verb_epochs, verb_rows = [], []
for yr, path in verb_logs.items():
    if not os.path.exists(path):
        print(f"Chưa có {path}"); continue
    rows, summary = parse_log_verb(path, yr)
    verb_epochs.extend(rows); verb_rows.append(summary)

df_verb_epochs  = pd.DataFrame(verb_epochs)
df_verb_summary = pd.DataFrame(verb_rows)
print(f"VerB epoch rows: {len(df_verb_epochs)}")
df_verb_summary

VerB epoch rows: 649


,test_year,best_epoch_by_ks,best_eval_acc,best_eval_recall_macro,best_eval_ks,best_eval_gm,final_eval_acc,final_eval_recall_macro,final_eval_ks,final_eval_gm
0,2018,46,78.62,71.48,42.96,69.93,75.36,67.74,35.48,65.88
1,2019,148,77.19,74.63,49.27,74.49,52.55,52.93,5.87,52.93
2,2020,5,79.02,68.97,37.93,67.21,43.58,39.24,21.52,38.67


### Nhận xét 7.2 — Kết quả VerB

Kết quả VerB cho thấy khi chọn best epoch theo eval KS, mô hình đạt chất lượng rất tốt ở các năm 2018 và 2019. Cụ thể, ở năm 2018, VerB đạt KS = 42.96%, Recallm = 71.48% và G-mean = 69.93%, đều ngang bằng hoặc nhỉnh hơn bài báo gốc. Ở năm 2019, VerB tiếp tục cho kết quả rất sát bài báo với KS = 49.27%, Recallm = 74.63% và G-mean = 74.49%, sai lệch rất nhỏ trên cả ba chỉ số chính. Điều này cho thấy thiết lập chọn best theo KS giúp mô hình bám sát mục tiêu đánh giá của bài báo hơn ở hai năm đầu.

Tuy nhiên, ở năm 2020, VerB suy giảm rõ hơn so với VerA, với KS = 37.93%, Recallm = 68.97% và G-mean = 67.21%. Dù vẫn đạt tiêu chí đề cương, kết quả này thấp hơn đáng kể so với bài báo và cũng thấp hơn VerA ở cùng năm. Ngoài ra, các chỉ số `final_eval_*` của VerB sau epoch tốt nhất giảm mạnh, đặc biệt ở năm 2019 và 2020, cho thấy mô hình rất nhạy với việc chọn epoch và không nên sử dụng kết quả ở epoch cuối cùng.

Nhìn chung, VerB có ưu thế rõ rệt ở các năm 2018 và 2019, nhưng độ ổn định giữa các năm không cao bằng VerA.

### 7.3 Bảng so sánh 3 chiều: Paper | VerA | VerB

Cột `diff_A` = VerA − Paper, `diff_B` = VerB − Paper, `diff_B_A` = VerB − VerA.

`diff_B_A` định lượng tác động của việc đổi early stopping và gradient accumulation.

In [108]:
paper = pd.DataFrame([
    {"test_year": 2018, "paper_acc": 77.60, "paper_rcl": 70.79, "paper_ks": 41.58, "paper_gm": 69.37},
    {"test_year": 2019, "paper_acc": 80.45, "paper_rcl": 75.44, "paper_ks": 50.88, "paper_gm": 74.89},
    {"test_year": 2020, "paper_acc": 79.43, "paper_rcl": 71.96, "paper_ks": 43.92, "paper_gm": 71.04},
])

# Lấy VerA metrics tại best_by_loss
vera_best = df_vera_summary[["test_year","best_eval_acc","best_eval_recall_macro",
                              "best_eval_ks","best_eval_gm"]].rename(columns={
    "best_eval_acc":           "vera_acc",
    "best_eval_recall_macro":  "vera_rcl",
    "best_eval_ks":            "vera_ks",
    "best_eval_gm":            "vera_gm",
})

# Lấy VerB metrics tại best_by_ks
verb_best = df_verb_summary[["test_year","best_eval_acc","best_eval_recall_macro",
                              "best_eval_ks","best_eval_gm"]].rename(columns={
    "best_eval_acc":           "verb_acc",
    "best_eval_recall_macro":  "verb_rcl",
    "best_eval_ks":            "verb_ks",
    "best_eval_gm":            "verb_gm",
})

cmp3 = paper.merge(vera_best, on="test_year").merge(verb_best, on="test_year")

for m in ["acc", "rcl", "ks", "gm"]:
    cmp3[f"diff_A_{m}"] = (cmp3[f"vera_{m}"] - cmp3[f"paper_{m}"]).round(2)
    cmp3[f"diff_B_{m}"] = (cmp3[f"verb_{m}"] - cmp3[f"paper_{m}"]).round(2)
    cmp3[f"diff_BA_{m}"] = (cmp3[f"verb_{m}"] - cmp3[f"vera_{m}"]).round(2)

# Hiển thị bảng chính
display(cmp3[["test_year",
    "paper_ks", "vera_ks", "diff_A_ks", "verb_ks", "diff_B_ks", "diff_BA_ks",
    "paper_rcl","vera_rcl","diff_A_rcl","verb_rcl","diff_B_rcl","diff_BA_rcl",
    "paper_gm", "vera_gm", "diff_A_gm", "verb_gm", "diff_B_gm", "diff_BA_gm",
]])

,test_year,paper_ks,vera_ks,diff_A_ks,verb_ks,diff_B_ks,diff_BA_ks,paper_rcl,vera_rcl,diff_A_rcl,verb_rcl,diff_B_rcl,diff_BA_rcl,paper_gm,vera_gm,diff_A_gm,verb_gm,diff_B_gm,diff_BA_gm
0,2018,41.58,39.07,-2.51,42.96,1.38,3.89,70.79,69.54,-1.25,71.48,0.69,1.94,69.37,68.48,-0.89,69.93,0.56,1.45
1,2019,50.88,47.59,-3.29,49.27,-1.61,1.68,75.44,73.79,-1.65,74.63,-0.81,0.84,74.89,73.51,-1.38,74.49,-0.40,0.98
2,2020,43.92,46.39,2.47,37.93,-5.99,-8.46,71.96,73.20,1.24,68.97,-2.99,-4.23,71.04,72.08,1.04,67.21,-3.83,-4.87


### Nhận xét 7.3 — So sánh Paper, VerA và VerB

Bảng so sánh ba chiều cho thấy cả VerA và VerB đều tiệm cận khá tốt với bài báo gốc, tuy nhiên mỗi phiên bản có ưu thế khác nhau theo từng năm kiểm thử. Ở năm 2018, VerB vượt VerA rõ rệt trên cả KS, Recallm và G-mean; đồng thời kết quả của VerB cũng nhỉnh hơn bài báo gốc. Ở năm 2019, VerB tiếp tục tốt hơn VerA, với mức sai lệch so với bài báo nhỏ hơn trên cả ba chỉ số chính. Điều này cho thấy việc chọn best epoch theo eval KS giúp mô hình phù hợp hơn với mục tiêu đánh giá ở hai năm này.

Ngược lại, ở năm 2020, VerA lại vượt trội hơn VerB khá rõ. VerA không chỉ tốt hơn VerB ở cả KS, Recallm và G-mean, mà còn đạt kết quả cao hơn bài báo gốc ở cả ba chỉ số. Trong khi đó, VerB giảm mạnh ở năm này, cho thấy tiêu chí chọn best theo KS không phải lúc nào cũng cho kết quả tốt nhất trên mọi tập kiểm thử.

Từ bảng này có thể rút ra rằng khác biệt giữa VerA và VerB phản ánh ảnh hưởng của chiến lược chọn best epoch. VerB phù hợp hơn ở các năm 2018 và 2019, còn VerA ổn định và hiệu quả hơn ở năm 2020. Như vậy, không có một tiêu chí chọn epoch tối ưu tuyệt đối cho mọi năm, và việc lựa chọn tiêu chí cần được xem xét cùng với đặc điểm của từng tập dữ liệu kiểm thử.

### 7.4 Kiểm tra tiêu chí đề cương

In [109]:
print("=" * 55)
print(f"  TIÊU CHÍ ĐỀ CƯƠNG: KS>=35%, Rcl+-5%, GM+-5%")
print("=" * 55)
for ver, ks_col, diff_col in [("VerA","vera_ks","A"), ("VerB","verb_ks","B")]:
    ks_ok  = (cmp3[ks_col] >= 35).any()
    rcl_ok = (cmp3[f"diff_{diff_col}_rcl"].abs() <= 5).all()
    gm_ok  = (cmp3[f"diff_{diff_col}_gm"].abs() <= 5).all()
    print(f"\n{ver}:")
    print(f"  KS >= 35% (ít nhất 1 năm) : {'OK' if ks_ok else 'FAIL'}")
    print(f"  Recallm trong +-5%        : {'OK' if rcl_ok else 'FAIL'}")
    print(f"  G-mean  trong +-5%        : {'OK' if gm_ok else 'FAIL'}")
    if ks_ok and rcl_ok and gm_ok:
        print(f"  => Đạt tiêu chí đề cương")
    else:
        print(f"  => Chưa đạt tiêu chí đề cương")

  TIÊU CHÍ ĐỀ CƯƠNG: KS>=35%, Rcl+-5%, GM+-5%

VerA:
  KS >= 35% (ít nhất 1 năm) : OK
  Recallm trong +-5%        : OK
  G-mean  trong +-5%        : OK
  => Đạt tiêu chí đề cương

VerB:
  KS >= 35% (ít nhất 1 năm) : OK
  Recallm trong +-5%        : OK
  G-mean  trong +-5%        : OK
  => Đạt tiêu chí đề cương


### Nhận xét 7.4 — Đối chiếu với tiêu chí đề cương

Kết quả kiểm tra cho thấy cả VerA và VerB đều đạt đầy đủ các tiêu chí chính đã đặt ra trong đề cương. Cụ thể, cả hai phiên bản đều có KS lớn hơn hoặc bằng 35% ở ít nhất một năm kiểm thử; đồng thời Recallm và G-mean đều nằm trong khoảng sai lệch ±5% so với bài báo gốc. Điều này có ý nghĩa quan trọng vì nó cho thấy thực nghiệm của nhóm không chỉ chạy được về mặt kỹ thuật mà còn đáp ứng được mục tiêu học thuật đã đề ra từ đầu.

Đối với VerA, việc đạt tiêu chí ở cả ba năm cho thấy thiết lập này có độ ổn định tốt. Đối với VerB, dù kết quả năm 2020 yếu hơn VerA, phiên bản này vẫn đạt đầy đủ tiêu chí đề cương và có ưu thế rõ hơn ở các năm 2018 và 2019. Như vậy, xét theo yêu cầu đề cương, cả hai thiết lập đều có thể xem là thành công.

Từ đây có thể khẳng định rằng phần tái hiện thực nghiệm cơ bản của ACRF-RNN đã hoàn thành đạt yêu cầu, và bộ dữ liệu cùng quy trình thực nghiệm hiện tại là đủ tin cậy để sử dụng trong báo cáo chính thức.

### Kết luận Giai đoạn 7 — Tổng hợp và so sánh kết quả

Tổng hợp từ các bảng kết quả cho thấy nhóm đã tái hiện thành công thực nghiệm cơ bản của mô hình ACRF-RNN trên bộ dữ liệu công khai. Cả VerA và VerB đều đạt tiêu chí đề cương, đồng thời cho kết quả khá sát với bài báo gốc trên các chỉ số quan trọng như KS, Recallm và G-mean. Điều này xác nhận rằng quy trình xử lý dữ liệu và thiết lập thực nghiệm của nhóm là hợp lý.

Về chi tiết, VerB cho kết quả tốt hơn ở các năm 2018 và 2019, tức là gần bài báo hơn ở đa số trường hợp; trong khi đó VerA lại ổn định hơn và vượt trội hơn ở năm 2020. Sự khác biệt này cho thấy chiến lược chọn best epoch có ảnh hưởng trực tiếp đến kết quả cuối cùng của mô hình. Nói cách khác, ngoài kiến trúc mô hình và dữ liệu đầu vào, cách lựa chọn epoch tốt nhất cũng là một yếu tố quan trọng cần được xem xét trong quá trình tái hiện thực nghiệm.

Nhìn chung, kết quả thu được là đủ mạnh để kết luận rằng phần thực nghiệm cơ bản của đề tài đã hoàn thành thành công. Đây là cơ sở quan trọng để sử dụng trong báo cáo chính thức, đồng thời cũng là nền tảng để nhóm tiếp tục chuyển sang các bước tiếp theo như hoàn thiện pipeline dữ liệu và triển khai hướng mở rộng trong đề tài.

### 7.5 Lưu kết quả

In [110]:
os.makedirs("/content/project_outputs", exist_ok=True)

if not df_vera_summary.empty:
    df_vera_epochs.to_csv("/content/project_outputs/epoch_vera.csv", index=False)
    df_vera_summary.to_csv("/content/project_outputs/summary_vera.csv", index=False)

if not df_verb_summary.empty:
    df_verb_epochs.to_csv("/content/project_outputs/epoch_verb.csv", index=False)
    df_verb_summary.to_csv("/content/project_outputs/summary_verb.csv", index=False)

if "cmp3" in dir():
    cmp3.to_csv("/content/project_outputs/compare_3way.csv", index=False)

print("Đã lưu kết quả")
!ls -lh /content/project_outputs/

Đã lưu kết quả
total 348K
-rw-r--r-- 1 root root  54K Jun  2 07:38 acrf-rnn.zip
drwxr-xr-x 2 root root 4.0K Jun  2 08:31 attention_graph
drwxr-xr-x 2 root root 4.0K Jun  2 07:57 checkpoints
-rw-r--r-- 1 root root  674 Jun  2 09:07 compare_3way.csv
-rw-r--r-- 1 root root  36K Jun  2 09:07 epoch_vera.csv
-rw-r--r-- 1 root root  29K Jun  2 09:07 epoch_verb.csv
-rw-r--r-- 1 root root  30K Jun  2 09:07 run_2018.log
-rw-r--r-- 1 root root  48K Jun  2 09:07 run_2019.log
-rw-r--r-- 1 root root  26K Jun  2 09:07 run_2020.log
-rw-r--r-- 1 root root  15K Jun  2 09:05 run_vera_2018.log
-rw-r--r-- 1 root root  24K Jun  2 09:06 run_vera_2019.log
-rw-r--r-- 1 root root  37K Jun  2 09:06 run_vera_2020.log
-rw-r--r-- 1 root root  15K Jun  2 07:57 run_vera_ckpt_2018.log
-rw-r--r-- 1 root root  421 Jun  2 09:07 summary_vera.csv
-rw-r--r-- 1 root root  324 Jun  2 09:07 summary_verb.csv


## Giai đoạn 8 — Nén và lưu Drive

In [111]:
!zip -r /content/project_outputs/acrf-rnn.zip /content/project_outputs/
!ls -lh /content/project_outputs/acrf-rnn.zip

import shutil, os
drive_save_path = '/content/drive/MyDrive/ACRF-RNN'
os.makedirs(drive_save_path, exist_ok=True)
shutil.copy('/content/project_outputs/acrf-rnn.zip',
            f'{drive_save_path}/acrf-rnn.zip')
print(f'Đã lưu vào Drive: {drive_save_path}/acrf-rnn.zip')

updating: content/project_outputs/ (stored 0%)
updating: content/project_outputs/summary_vera.csv (deflated 48%)
updating: content/project_outputs/epoch_vera.csv (deflated 73%)
updating: content/project_outputs/run_vera_2018.log (deflated 75%)
updating: content/project_outputs/epoch_verb.csv (deflated 68%)
updating: content/project_outputs/summary_verb.csv (deflated 45%)
updating: content/project_outputs/compare_3way.csv (deflated 52%)
updating: content/project_outputs/run_vera_2020.log (deflated 81%)
updating: content/project_outputs/attention_graph/ (stored 0%)
updating: content/project_outputs/run_vera_2019.log (deflated 77%)
updating: content/project_outputs/run_2018.log (deflated 85%)
updating: content/project_outputs/run_2019.log (deflated 86%)
updating: content/project_outputs/run_2020.log (deflated 84%)
  adding: content/project_outputs/checkpoints/ (stored 0%)
  adding: content/project_outputs/checkpoints/vera_best_test_year_2018.pt (deflated 8%)
  adding: content/project_outp